In [ ]:
import pandas as pd
import numpy as np
import torch
from utils.utils_dataset import prepare_non_iid_federated_dataset_for_clustering, prepare_iid_federated_dataset_for_clustering
import os
from utils.utils_train import train_model_sequentially_loader, train_model_sequentially_prealocated, train_model_prealocated, train_model_loader
from utils.utils_plots import plot_first_feature_combinations, plot_cluster_density_heatmap, plot_results_acc
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score,f1_score
from collections import Counter
from scipy.stats import chi2

In [ ]:
from model.eFDPC import eFDPC
from model.clustering_operations import ClusteringOps
from model.removal_mechanism import RemovalMechanism 
from model.merging_mechanism import MergingMechanism
from model.math_operations import MathOps
from model.consequence_operations import ConsequenceOps 
from model.model_operations import ModelOps
from model.federated_operations import FederalOps
from model.hierarchical_clustering import HierarchicalClustering
from model.shuffler_privacy import ShufflerPrivacy

print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)
torch.set_num_threads(os.cpu_count())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from pathlib import Path
import random, time
import numpy as np
import pandas as pd
import torch
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = "none"

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def save_fig_vector(fig, out_base: Path, also_png: bool = False, dpi_png: int = 300):
    """
    Saves vector PDF (+ SVG). PNG optional.
    out_base: path without suffix, e.g. Path(".../Spiral_eFDPC_rep0")
    """
    ensure_dir(out_base.parent)
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    if also_png:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=dpi_png)


In [ ]:
def determine_cluster_labels(y_true, cluster_assignments):
    cluster_labels = {}
    for cluster in np.unique(cluster_assignments):
        cluster_indices = cluster_assignments == cluster
        most_common_label = Counter(y_true[cluster_indices]).most_common(1)[0][0]
        cluster_labels[cluster] = most_common_label
    return cluster_labels

def remap_assignments(assignments, cluster_labels):
    remapped_labels = np.array([cluster_labels[cluster] for cluster in assignments])
    return remapped_labels

In [ ]:
import psutil, torch, os, math
from torch.utils.data import DataLoader, TensorDataset

def preload_or_loader(X, y, batch_size, device, safety=0.6):
    """
    Decide automatically:
      • if X,y fit in GPU  -> move once to GPU, return (X_gpu, y_gpu)
      • elif fit in CPU RAM -> pin-memory, return (X_pin, y_pin)
      • else                -> return DataLoader
    """
    X = torch.as_tensor(X)
    y = torch.as_tensor(y)

    n_bytes = X.element_size() * X.numel() + y.element_size() * y.numel()

    if device.type == "cuda":
        free_gpu = torch.cuda.mem_get_info()[0] * safety
        if n_bytes <= free_gpu:
            return (X.to(device, non_blocking=True),
                    y.to(device, non_blocking=True))

    free_ram = psutil.virtual_memory().available * safety
    if n_bytes <= free_ram:
        return (X.pin_memory(), y.pin_memory())

    ds = TensorDataset(X, y)
    loader = DataLoader(ds,
                        batch_size=batch_size,
                        shuffle=True,
                        pin_memory=True,
                        num_workers=min(os.cpu_count(), 8),
                        persistent_workers=True,
                        prefetch_factor=4)
    return loader

def choose_src(X, y, device, batch_size, safety=0.6):
    """Return (X_buf,y_buf)  or  a DataLoader."""
    X, y = torch.as_tensor(X), torch.as_tensor(y)
    n_bytes = X.element_size()*X.numel() + y.element_size()*y.numel()

    if device.type == "cuda":
        if n_bytes <= torch.cuda.mem_get_info()[0] * safety:
            return (X.to(device, non_blocking=True),
                    y.to(device, non_blocking=True))

    if n_bytes <= psutil.virtual_memory().available * safety:
        return (X.pin_memory(), y.pin_memory())

    return DataLoader(TensorDataset(X, y),
                      batch_size=batch_size,
                      shuffle=True,
                      pin_memory=True,
                      num_workers=min(8, os.cpu_count()),
                      persistent_workers=True,
                      prefetch_factor=4)
    import psutil, torch

def memory_check(train_datasets, device: torch.device, safety: float = 0.6) -> bool:
    total_bytes = sum(
        X.numel() * X.element_size() + y.numel() * y.element_size()
        for (X, y) in (ds.tensors for ds in train_datasets)
    )

    if device.type == "cuda":
        free = torch.cuda.mem_get_info()[0] * safety
    else:
        free = psutil.virtual_memory().available * safety

    return total_bytes > free


In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

def plot_model_dendrogram(mu, S, n,
                          merging_weight=1.0,
                          linkage_method='average',
                          cut_k=None,
                          cut_dist=None):
    """
    mu  : (C, D) array of cluster centers
    S   : (C, D) array of diagonal scatters
    n   : (C,)   array of counts
    Returns (fig, groups) where groups[i] is the flat‐cluster ID of i.
    """
    C, D = mu.shape
    iu, ju = np.triu_indices(C, 1)
    mu_diff = mu[iu] - mu[ju]
    n_sum   = n[iu] + n[ju]
    Sigma12 = (S[iu] + S[ju]) / n_sum[:,None]
    dM2     = np.sum(mu_diff**2 / Sigma12, axis=1)

    Dmat = np.zeros((C, C))
    Dmat[iu, ju] = Dmat[ju, iu] = dM2

    condensed = squareform(Dmat)
    Z = linkage(condensed, method=linkage_method)

    fig, ax = plt.subplots(figsize=(6, 4))
    dendrogram(Z, ax=ax, labels=[f"C{i}" for i in range(C)], leaf_rotation=90)
    ax.set_title("Cluster dendrogram")
    plt.tight_layout()

    if cut_k is not None:
        groups = fcluster(Z, t=cut_k, criterion="maxclust")
    elif cut_dist is not None:
        groups = fcluster(Z, t=cut_dist, criterion="distance")
    else:
        groups = None

    return fig, groups


In [ ]:
from scipy.spatial.distance import cdist
def predict_clusters(X, centers):

    distances = cdist(X, centers, metric='euclidean')

    return np.argmin(distances, axis=1)



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

def _safe_len_ds(ds):
    try:
        return int(len(ds))
    except Exception:
        try:
            return int(ds.tensors[0].shape[0])
        except Exception:
            return None

def summarize_federated_split(train_datasets, test_dataset, X, y, num_clients, batch_size):
    """
    Returns a dict with dataset/split statistics.
    """
    N = int(len(y))
    D = int(X.shape[1]) if hasattr(X, "shape") else int(np.asarray(X).shape[1])

    train_sizes = [int(_safe_len_ds(ds)) for ds in train_datasets]
    test_size = int(_safe_len_ds(test_dataset))

    y_np = y.detach().cpu().numpy() if torch.is_tensor(y) else np.asarray(y)
    uniq, cnt = np.unique(y_np, return_counts=True)
    label_hist = dict(zip([int(u) for u in uniq], [int(c) for c in cnt]))

    out = {
        "N": N,
        "D": D,
        "num_clients": int(num_clients),
        "batch_size": int(batch_size),
        "train_total": int(sum(train_sizes)),
        "test_N": int(test_size),
        "train_min": int(np.min(train_sizes)) if train_sizes else np.nan,
        "train_max": int(np.max(train_sizes)) if train_sizes else np.nan,
        "train_mean": float(np.mean(train_sizes)) if train_sizes else np.nan,
        "train_std": float(np.std(train_sizes, ddof=0)) if train_sizes else np.nan,
        "train_sizes": ";".join(map(str, train_sizes)),
        "label_hist": str(label_hist),
    }
    return out

def append_or_overwrite_csv(path: Path, df: pd.DataFrame, overwrite: bool):
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or (not path.exists()):
        df.to_csv(path, index=False)
    else:
        df.to_csv(path, index=False, mode="a", header=False)


In [ ]:
from pathlib import Path
import pandas as pd

def upsert_dataset_info_csv(out_root: str | Path, row: dict, filename: str = "dataset_info.csv"):
    """
    One global CSV.
    ONE ROW PER DATASET (+ split if you want).
    If the row exists -> overwrite it (update).
    Else -> append it.
    """
    out_root = Path(out_root)
    tables_dir = out_root / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    csv_path = tables_dir / filename

    KEY_COLS = ["dataset"]

    new_df = pd.DataFrame([row])

    if csv_path.exists():
        df = pd.read_csv(csv_path)

        for c in new_df.columns:
            if c not in df.columns:
                df[c] = pd.NA
        for c in df.columns:
            if c not in new_df.columns:
                new_df[c] = pd.NA
        new_df = new_df[df.columns]

        mask = pd.Series([True] * len(df))
        for k in KEY_COLS:
            mask &= (df[k].astype(str) == str(row[k]))
        df = df.loc[~mask].copy()

        df = pd.concat([df, new_df], ignore_index=True)
    else:
        df = new_df

    if "dataset" in df.columns:
        df = df.sort_values(["dataset"]).reset_index(drop=True)

    df.to_csv(csv_path, index=False)
    return csv_path


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, accuracy_score, f1_score, silhouette_score

def eval_labels(X_test, y_true, y_pred):
    out = {}
    out["ARI"] = float(adjusted_rand_score(y_true, y_pred))
    out["NMI"] = float(normalized_mutual_info_score(y_true, y_pred))
    out["ACC"] = float(accuracy_score(y_true, y_pred))
    out["F1"]  = float(f1_score(y_true, y_pred, average="macro"))
    out["K_pred"] = int(len(np.unique(y_pred)))
    return out


In [ ]:
@torch.no_grad()
def scale_stats(model, factor: float):
    """Scale all weight-like stats so we can do EMA without growth."""
    C = int(model.c)
    f = float(factor)

    model.n[:C].mul_(f)
    model.S[:C].mul_(f)

    if hasattr(model, "n_glo"):   model.n_glo.mul_(f)
    if hasattr(model, "S_glo"):   model.S_glo.mul_(f)
    if hasattr(model, "var_glo"): model.var_glo.mul_(f)

@torch.no_grad()
def normalize_mass(model, total_mass: float = 1.0, eps: float = 1e-12):
    """
    Force sum(n) to a constant. n becomes a *weight* not a true count.
    Keep ellipse sizes stable by scaling S together with n.
    """
    C = int(model.c)
    n = model.n[:C].clamp_min(0.0)
    s = n.sum().clamp_min(eps)
    f = float(total_mass) / float(s)

    model.n[:C].mul_(f)
    model.S[:C].mul_(f)


In [ ]:
def _banner(s):
    s = str(s)
    print("\n\n" + "########################### ")
    print( "######### " + s + " ######### ")
    print("########################### ")
    
def _print_metrics(m):
    keys = ["method", "ACC", "ARI", "NMI", "F1", "K_pred", "K_micro", "K_centers", "time_sec", "_error"]
    out = []
    for k in keys:
        if k in m:
            out.append((k, m[k]))
    print(out)

def canon_method(s):
    s = "" if s is None else str(s)
    m = {
        "Our_eFDPC_centers": "Our_eFDPC_centers",
        "Our_eFDPC_group":   "Our_eFDPC_group",
        "Our_eFDPC":         "Our_eFDPC",
        "FKDC":              "FKDC",
        "kFED":              "kFED",
        "MUFC":              "MUFC",
        "FED_FCM":           "FED_FCM",
        "LLF_FCM":           "LLF_FCM",
        "LLF_CM":            "LLF_CM",
        "FED-FCM":           "FED_FCM",
        "LLF-FCM":           "LLF_FCM",
        "LLF-CM":            "LLF_CM",
    }
    return m.get(s, s)


In [ ]:
import os
RUN_BASELINES = os.path.isdir("related_methods")

In [ ]:
def run_benchmark_experiment(
    X, y,
    local_model_params, federated_model_params,
    num_clients, batch_size, num_clusters,
    rounds: int,
    num_repetitions: int,
    flag_noniid: bool,
    name: str,
    debug_flag: int = 0,
    base_seed: int = 1,
    out_root: str = ".benchmarks",
    flag_save_img: bool = False,
    flag_axis: bool = False,
    save_vec: bool = True,
    overwrite_dataset_csv: bool = True,
    dataset_key: str | None = None,
    alpha = 0.9,
):
    out_root = Path(out_root)
    split_name = "noniid" if flag_noniid else "iid"

    dataset_key = name if dataset_key is None else str(dataset_key)
    run_name = str(name)

    tables_dir = out_root / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    info_row = {
        "dataset": dataset_key,
        "split": split_name,
        "N": int(X.shape[0]),
        "D": int(X.shape[1]),
        "clients": int(num_clients),
        "K_true": int(num_clusters),
    }
    info_csv = upsert_dataset_info_csv(out_root, info_row, filename="dataset_info.csv")
    print(f"Updated dataset info: {info_csv}")

    rows = []

    for rep in range(num_repetitions):
        seed = base_seed + rep
        set_all_seeds(seed)

        if flag_noniid:
            train_loaders, train_datasets, test_loader, test_dataset, all_data, train_data = \
                prepare_non_iid_federated_dataset_for_clustering(X, y, num_clients=num_clients, alpha=alpha,  batch_size=batch_size, min_samples_per_client=1)
        else:
            train_loaders, train_datasets, test_loader, test_dataset, all_data, train_data = \
                prepare_iid_federated_dataset_for_clustering(X, y, num_clients, batch_size=batch_size)

        common = {
            "dataset": dataset_key,
            "run": run_name,
            "split": split_name,
            "rep": rep,
            "seed": seed,
            "clients": int(num_clients),
            "rounds": int(rounds),
            "K_true": int(num_clusters),
        }

        _banner(f"{dataset_key} rep={rep} | Our_eFDPC")
        ms = run_efdpc_experiment(
            train_datasets=train_datasets,
            train_loaders=train_loaders,
            test_dataset=test_dataset,
            num_clients=num_clients,
            num_clusters=num_clusters,
            rounds=rounds,
            batch_size=batch_size,
            local_model_params=local_model_params,
            federated_model_params=federated_model_params,
            name=f"{run_name}_eFDPC_rep{rep}",
            profiler_flag=False,
            debug_flag=debug_flag,
            flag_save_img=flag_save_img,
        )
        ms = ms if isinstance(ms, (list, tuple)) else [ms]
        for m in ms:
            m.update(common)
            m["method"] = canon_method(m.get("method", ""))
            _print_metrics(m)
            rows.append(m)
        
    
        if RUN_BASELINES:
            _banner(f"{dataset_key} rep={rep} | FKDC")
            m = run_fkdc_experiment(
                train_datasets, test_dataset,
                num_clients, num_clusters, rounds,
                name=f"{run_name}_FKDC_rep{rep}",
                random_clients_seed=seed,
                fig_dir=None,
                save_vec=save_vec,
                flag_save_img=flag_save_img,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

            _banner(f"{dataset_key} rep={rep} | kFED")
            m = run_kfed_experiment(
                train_datasets, test_dataset, num_clients, num_clusters, rounds,
                name=f"{run_name}_kFED_rep{rep}",
                random_clients_seed=seed,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

            _banner(f"{dataset_key} rep={rep} | MUFC")
            m = run_mufc_experiment(
                train_datasets, test_dataset, num_clients, num_clusters, rounds,
                name=f"{run_name}_MUFC_rep{rep}",
                random_clients_seed=seed,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

            _banner(f"{dataset_key} rep={rep} | FED_FCM")
            m = run_fed_fcm_experiment(
                train_datasets, test_dataset, num_clients, num_clusters, rounds,
                name=f"{run_name}_FED_FCM_rep{rep}",
                random_clients_seed=seed,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

            _banner(f"{dataset_key} rep={rep} | LLF_FCM")
            m = run_llf_fcm_experiment(
                train_datasets, test_dataset, num_clients, num_clusters, rounds,
                name=f"{run_name}_LLF_FCM_rep{rep}",
                random_clients_seed=seed,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

            _banner(f"{dataset_key} rep={rep} | LLF_CM")
            m = run_llf_cm_experiment(
                train_datasets, test_dataset, num_clients, num_clusters, rounds,
                name=f"{run_name}_LLF_CM_rep{rep}",
                random_clients_seed=seed,
            )
            m.update(common); m["method"] = canon_method(m.get("method", "")); _print_metrics(m); rows.append(m)

    df = pd.DataFrame(rows)

    dataset_csv = tables_dir / f"{run_name}.csv"
    if overwrite_dataset_csv and dataset_csv.exists():
        dataset_csv.unlink()
    df.to_csv(dataset_csv, index=False)
    print(f"Saved dataset CSV: {dataset_csv}")

    return df, dataset_csv


In [ ]:
import sys
if RUN_BASELINES:
    try:
        sys.path.append('related_methods/Clustering_LLF-FCM')
        from FedFuzzClus.ffch import run_ffch_experiment
        from FedFuzzClus.fch import run_fch_experiment
    except Exception:
        RUN_BASELINES = False
import numpy as np
import time
from pathlib import Path

def run_llf_fcm_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=42,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    start_time = time.time()

    client_params = {
        "centroid_seed": random_clients_seed,
        "num_clusters": num_clusters,
        "lambda_factor": 2.0
    }
    server_params = {
        "epsilon": 0.001,
        "max_number_rounds": rounds,
        "num_clusters": num_clusters
    }

    client_data = []
    y_chunks = []
    for i in range(num_clients):
        X_numpy = train_datasets[i].tensors[0].detach().cpu().numpy()
        client_data.append(X_numpy)
        y_chunks.append(np.zeros(X_numpy.shape[0]))

    server, clients, final_centers = run_ffch_experiment(
        num_clients, num_clients,
        client_data, y_chunks,
        client_params, server_params,
        random_clients=True,
        random_clients_seed=random_clients_seed
    )

    execution_time = time.time() - start_time
    final_centers = np.asarray(final_centers)

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "LLF_FCM"

    internal_save = flag_save_img and (fig_dir is None)
    if flag_plot_acc:
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
from pathlib import Path
import numpy as np
import time

def run_llf_cm_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=0,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    start_time = time.time()

    client_params = {
        "centroid_seed": random_clients_seed,
        "num_clusters": num_clusters
    }
    server_params = {
        "epsilon": 0.001,
        "max_number_rounds": rounds,
        "num_clusters": num_clusters
    }

    client_data = []
    y_chunks = []
    for i in range(num_clients):
        X_numpy = train_datasets[i].tensors[0].detach().cpu().numpy()

        if X_numpy.ndim == 1:
            X_numpy = X_numpy.reshape(-1, 1)

        client_data.append(X_numpy.astype(np.float64, copy=False))
        y_chunks.append(np.zeros(X_numpy.shape[0], dtype=np.float64))

    final_centers = None
    last_err = None

    for attempt, (rand_flag, seed_offset) in enumerate([
        (True,  0),
        (False, 0),
        (False, 99991),
    ]):
        try:
            server, clients, centers = run_fch_experiment(
                num_clients, num_clients,
                client_data, y_chunks,
                client_params, server_params,
                random_clients=rand_flag,
                random_clients_seed=random_clients_seed + seed_offset
            )
            final_centers = np.asarray(centers)
            break
        except TypeError as e:
            last_err = e

    if final_centers is None:
        exec_time = time.time() - start_time
        X_test = test_dataset.tensors[0].detach().cpu().numpy()
        y_true = test_dataset.tensors[1].detach().cpu().numpy()
        return {
            "ARI": np.nan, "NMI": np.nan, "ACC": np.nan, "F1": np.nan,
            "K_pred": np.nan, "Silhouette": np.nan,
            "time_sec": float(exec_time),
            "method": "LLF_CM",
            "_error": str(last_err),
        }

    execution_time = time.time() - start_time

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "LLF_CM"

    internal_save = flag_save_img and (fig_dir is None)
    
    if flag_plot_acc:
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def debug_snapshot_model(
    model,
    *,
    tag: str = "",
    K_show: int = 10,
    print_flag: bool = True,
    plot_flag: bool = False,
    plot_all_centers: bool = True,
    dims: tuple[int, int] = (0, 1),
):
    """
    eFDPC debug snapshot:
      - prints: c, c_eff, n_glo/var_glo (if present), and a top-K table sorted by n
      - plots: all centers in dims=(d0,d1) if feature_dim>=2
    """
    C = int(getattr(model, "c", 0))
    if C <= 0:
        if print_flag:
            print(f"[{tag}] c=0")
        return

    mu = model.mu[:C].detach().to("cpu")
    n  = model.n[:C].detach().to("cpu").view(-1)
    c_eff = int((n > 0).sum().item())

    S = getattr(model, "S", None)
    S_tr = None
    if S is not None:
        S0 = S[:C].detach().to("cpu")
        if S0.ndim == 2:
            S_tr = S0.sum(dim=1)
        elif S0.ndim == 3:
            S_tr = torch.diagonal(S0, dim1=1, dim2=2).sum(dim=1)

    n_glo = getattr(model, "n_glo", None)
    var_glo = getattr(model, "var_glo", None)

    if print_flag:
        header = f"[{tag}] c={C} c_eff={c_eff}"
        if n_glo is not None:
            try:
                header += f" n_glo={float(torch.as_tensor(n_glo).detach().cpu())}"
            except Exception:
                header += f" n_glo={n_glo}"
        if var_glo is not None:
            try:
                header += f" var_glo={float(torch.as_tensor(var_glo).detach().cpu())}"
            except Exception:
                header += f" var_glo={var_glo}"
        print(header)

        K = int(min(max(1, K_show), C))
        order = torch.argsort(n, descending=True)
        top = order[:K].tolist()

        print("  topK by n:")
        if S_tr is None:
            print("    idx |      n")
            for idx in top:
                print(f"    {idx:3d} | {float(n[idx]):8.3f}")
        else:
            print("    idx |      n |  tr(S)")
            for idx in top:
                print(f"    {idx:3d} | {float(n[idx]):8.3f} | {float(S_tr[idx]):10.3e}")

        if S_tr is not None:
            nz = (n > 0)
            if nz.any():
                tr_vals = S_tr[nz]
                print(
                    f"  tr(S) nz: min={float(tr_vals.min()):.3e} "
                    f"med={float(tr_vals.median()):.3e} max={float(tr_vals.max()):.3e}"
                )

    if plot_flag:
        d0, d1 = int(dims[0]), int(dims[1])
        if mu.shape[1] <= max(d0, d1):
            print(f"[{tag}] plot skipped: feature_dim={mu.shape[1]} < {max(d0,d1)+1}")
            return

        x = mu[:, d0].numpy()
        y = mu[:, d1].numpy()

        plt.figure()
        if plot_all_centers:
            plt.scatter(x, y, s=10)
        else:
            K = int(min(max(1, K_show), C))
            order = torch.argsort(n, descending=True)[:K]
            xx = mu[order, d0].numpy()
            yy = mu[order, d1].numpy()
            plt.scatter(xx, yy, s=25)

        plt.xlabel(f"mu[{d0}]")
        plt.ylabel(f"mu[{d1}]")
        plt.title(tag)
        plt.tight_layout()
        plt.show()

In [ ]:
import copy
from federated_server import efdpc_server


In [ ]:

from pathlib import Path
import time
import torch
import numpy as np

def run_efdpc_experiment(
    rounds,
    num_clients,
    local_model_params,
    federated_model_params,
    train_datasets,
    train_loaders,
    test_dataset,
    batch_size,
    num_clusters,
    profiler_flag,
    debug_flag,
    name=None,
    flag_save_img=False,
    flag_axis=True,
    flag_history_eval=False,
):
    import cProfile
    if profiler_flag:
        pr = cProfile.Profile()
        pr.enable()

    start_time = time.time()

    flag_use_loader = memory_check(train_datasets, local_model_params["device"])

    out = efdpc_server(
        train_datasets=train_datasets,
        train_loaders=train_loaders,
        num_clients=num_clients,
        num_clusters=num_clusters,
        rounds=rounds,
        batch_size=batch_size,
        local_model_params=local_model_params,
        federated_model_params=federated_model_params,
        flag_use_loader=flag_use_loader,
        debug_flag=debug_flag,
        flag_history=bool(flag_history_eval),
    )

    models = out if isinstance(out, list) else [out]
    if not flag_history_eval:
        models = [models[-1]]

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test_numpy = X_test_tensor.detach().cpu().numpy()
    y_test_numpy = y_test_tensor.detach().cpu().numpy()

    all_metrics = []
    for t, federated_model in enumerate(models, start=1):
        times_sec = time.time() - start_time

        if debug_flag == 22:
            federated_model.federal_agent.plot_saved_shufflers_centers(title_prefix=f"Shuffler {name}")

        num_federated_clusters = int(torch.sum(federated_model.n[:federated_model.c] > 0).cpu().item())

        hc = federated_model.hierarch
        final_centers = federated_model.mu[:federated_model.c].detach().cpu().numpy()

        dbg = debug_flag in [14, 19]
        if dbg:
            hc.plot_kappa_graph_Sglo(use_groups=True)
            hc.plot_kappa_components_Sglo()
            hc.plot_decision_panels()
            hc.plot_centers_alpha()
            out_dir = Path(".Images/DPC")
            out_dir.mkdir(parents=True, exist_ok=True)

            hc.plot_decision_graph(name=name, out_dir=out_dir, ext="pdf", show=True)
            hc.plot_imp_delta_elbow(name=name, out_dir=out_dir, ext="pdf", show=True)
            hc.plot_all_cluster_groupings(name=name, out_dir=out_dir, ext="pdf", show=True)

        if debug_flag == 18:
            C = federated_model.c
            mu = federated_model.mu[:C].cpu().detach().numpy()
            S  = federated_model.S[:C].cpu().detach().numpy()
            n  = federated_model.n[:C].cpu().detach().numpy()
            fig_dendro, groups = plot_model_dendrogram(
                mu, S, n,
                merging_weight=federated_model.kappa_n,
                linkage_method="average",
                cut_k=5
            )
            fig_dendro.suptitle("Federated clusters dendrogram (flat k=5)")

        if debug_flag in [21]:
            fig = plot_first_feature_combinations(
                (X_test_tensor, y_test_tensor),
                model=federated_model,
                num_sigma=2,
                N_max=0,
                flag_unlabeled=True,
            )
            if flag_save_img and fig is not None:
                Path(".Images/Clustering").mkdir(parents=True, exist_ok=True)
                suffix = f"_r{t}" if flag_history_eval else ""
                fig.savefig(f".Images/Clustering/{name}{suffix}.svg", bbox_inches="tight")
                fig.savefig(f".Images/Clustering/{name}{suffix}.png", bbox_inches="tight", dpi=300)
                fig.savefig(f".Images/Clustering/{name}{suffix}.pdf", bbox_inches="tight")

        if debug_flag == 13:
            plot_cluster_density_heatmap(federated_model)

        if debug_flag:
            print(f"[round_eval {t}/{len(models)}] Number of clusters in federated model: {num_federated_clusters}")

        center_ids = hc.centers[0].detach().cpu().numpy().astype(np.int64)
        K_centers = int(center_ids.shape[0])

        center_coords = federated_model.mu[center_ids].detach().cpu().numpy()
        center_assignments = predict_clusters(X_test_numpy, center_coords)
        center_labels = determine_cluster_labels(y_test_numpy, center_assignments)
        y_pred_center = remap_assignments(center_assignments, center_labels)

        m_center = eval_labels(X_test_numpy, y_test_numpy, y_pred_center)
        m_center["time_sec"]  = float(times_sec)
        m_center["method"]    = "Our_eFDPC_centers"
        m_center["K_micro"]   = int(num_federated_clusters)
        m_center["K_centers"] = int(K_centers)
        if flag_history_eval:
            m_center["round_idx"] = int(t)

        if flag_plot_acc:
            fig = plot_results_acc(
                X_test_numpy, y_test_numpy,
                center_coords, y_pred_center,
                f"{name}_centers" + (f"_r{t}" if flag_history_eval else ""),
                times_sec,
                None, m_center["ARI"], m_center["NMI"], m_center["ACC"],
                flag_save_img=flag_save_img,
            )

        cluster_to_group = hc.cluster_to_group[:federated_model.c].detach().cpu().numpy()

        cluster_assignments = predict_clusters(X_test_numpy, final_centers)
        group_assignments   = cluster_to_group[cluster_assignments]

        group_labels = determine_cluster_labels(y_test_numpy, group_assignments)
        y_pred_group = remap_assignments(group_assignments, group_labels)

        m_group = eval_labels(X_test_numpy, y_test_numpy, y_pred_group)
        m_group["time_sec"]  = float(times_sec)
        m_group["method"]    = "Our_eFDPC_group"
        m_group["K_micro"]   = int(num_federated_clusters)
        m_group["K_centers"] = int(K_centers)
        if flag_history_eval:
            m_group["round_idx"] = int(t)

        group_centers = hc.group_centers.detach().cpu().numpy()
        if flag_plot_acc:
            fig = plot_results_acc(
                X_test_numpy, y_test_numpy,
                group_centers, y_pred_group,
                f"{name}_group" + (f"_r{t}" if flag_history_eval else ""),
                times_sec,
                None, m_group["ARI"], m_group["NMI"], m_group["ACC"],
                flag_save_img=flag_save_img,
            )

        all_metrics.extend([m_center, m_group])

    if profiler_flag:
        pr.disable()
        pr.print_stats(sort="cumtime")

    return all_metrics


In [ ]:
import numpy as np
import time
import sys
from pathlib import Path

if RUN_BASELINES:
    try:
        sys.path.append('related_methods/Clustering_federated_clustering')
        from global_learner import GlobalClusterer
        from local_learners import FuzzyCMeansClient
    except Exception:
        RUN_BASELINES = False

def run_fed_fcm_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=0,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    start_time = time.time()

    data_dim = train_datasets[0].tensors[0].shape[1]

    client_data = [
        train_datasets[i].tensors[0].detach().cpu().numpy()
        for i in range(num_clients)
    ]

    clients = [FuzzyCMeansClient(client_data[i], num_clusters) for i in range(num_clients)]
    global_clusterer = GlobalClusterer(clients, num_clusters, data_dim, max_rounds=rounds)

    global_clusterer.fit()
    final_centers = np.asarray(global_clusterer.cluster_centers)

    execution_time = time.time() - start_time

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "FED_FCM"

    internal_save = flag_save_img and (fig_dir is None)
    if flag_plot_acc:
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
from pathlib import Path
import time
import numpy as np
import sys

if RUN_BASELINES:
    try:
        sys.path.append("related_methods/FKDC")
        import fkdc_utils as fkdc_utils
    except Exception:
        RUN_BASELINES = False

def run_fkdc_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=42,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    
    np.random.seed(random_clients_seed)

    start_time = time.time()

    client_data = [
        train_datasets[i].tensors[0].detach().cpu().numpy()
        for i in range(num_clients)
    ]

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    arr_fkdc, final_centers, params = fkdc_utils.results_from_arrays(
        client_data_list=client_data,
        full_data=X_test,
        true_label=y_true,
        num_clusters=num_clusters,
        eachlable=None,
        order=list(range(num_clients)),
        dp_b=1/10,
        snn_k=20,
    )

    final_centers = np.asarray(final_centers)

    execution_time = time.time() - start_time

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "FKDC"

    metrics["fkdc_local_k"] = float(params[0]) if len(params) > 0 else None
    metrics["fkdc_server_k"] = float(params[1]) if len(params) > 1 else None
    metrics["fkdc_snn_k"] = float(params[2]) if len(params) > 2 else None

    internal_save = flag_save_img and (fig_dir is None)

    if flag_plot_acc:
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
from pathlib import Path
import time
import numpy as np
import sys

if RUN_BASELINES:
    try:
        sys.path.append('related_methods/Clustering_kfed')
        from fkms.algorithms import kfed
    except Exception:
        RUN_BASELINES = False

def run_kfed_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=42,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    start_time = time.time()

    client_data = [
        train_datasets[i].tensors[0].detach().cpu().numpy()
        for i in range(num_clients)
    ]

    local_estimates, final_centers = kfed(
        x_dev=client_data,
        dev_k=num_clusters,
        k=num_clusters,
    )
    final_centers = np.asarray(final_centers)

    execution_time = time.time() - start_time

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "kFED"

    internal_save = flag_save_img and (fig_dir is None)
    
    if flag_plot_acc:
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
from pathlib import Path
import time
import numpy as np
import torch
import sys

if RUN_BASELINES:
    try:
        sys.path.append('related_methods/Clustering_mufc')
        from mufc_fun import federated_clustering_simple
    except Exception:
        RUN_BASELINES = False

def run_mufc_experiment(
    train_datasets, test_dataset,
    num_clients, num_clusters, rounds,
    name,
    random_clients_seed=0,
    fig_dir: Path | None = None,
    save_vec: bool = True,
    flag_save_img: bool = False,
):
    np.random.seed(random_clients_seed)
    torch.manual_seed(random_clients_seed)

    start_time = time.time()

    dataset = {
        "full_data": torch.cat([train_datasets[i].tensors[0] for i in range(num_clients)])
                      .detach().cpu().numpy()
    }
    for i in range(num_clients):
        dataset[f"client_{i}"] = train_datasets[i].tensors[0].detach().cpu().numpy()

    results = federated_clustering_simple(
        dataset=dataset,
        num_clusters=num_clusters,
        num_clients=num_clients,
        split="iid",
        k_prime=num_clusters,
        max_iters=rounds,
    )

    final_centers = np.asarray(results["final_clusters"])
    execution_time = time.time() - start_time

    X_test_tensor, y_test_tensor = test_dataset.tensors
    X_test = X_test_tensor.detach().cpu().numpy()
    y_true = y_test_tensor.detach().cpu().numpy()

    test_assignments = predict_clusters(X_test, final_centers)
    cluster_labels = determine_cluster_labels(y_true, test_assignments)
    y_pred = remap_assignments(test_assignments, cluster_labels)

    metrics = eval_labels(X_test, y_true, y_pred)
    metrics["time_sec"] = float(execution_time)
    metrics["method"] = "MUFC"

    internal_save = flag_save_img and (fig_dir is None)

    if flag_plot_acc:
        
        fig = plot_results_acc(
            X_test, y_true, final_centers, y_pred, name,
            execution_time,
            None, metrics["ARI"], metrics["NMI"], metrics["ACC"],
            flag_save_img=internal_save,
            seed=random_clients_seed,
        )

    if flag_save_img and (fig_dir is not None) and save_vec and (fig is not None):
        out_base = Path(fig_dir) / name
        save_fig_vector(fig, out_base, also_png=True)

    return metrics


In [ ]:
import requests

if 0:
    
    
    response = requests.get("https://cs.uef.fi/sipu/datasets/unbalance2.txt")
    with open(".datasets/Unbalance2.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/unbalance2.pa")
    with open(".datasets/unbalance2.pa", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/overlap.txt")
    with open(".datasets/Overlap.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/overlap.pa")
    with open(".datasets/overlap.pa", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/skewed.txt")
    with open(".datasets/Skewed.txt", "wb") as f:
        f.write(response.content)
        
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/skewed.pa")
    with open(".datasets/skewed.pa", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/asymmetric.txt")
    with open(".datasets/Asymmetric.txt", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/asymmetric.pa")
    with open(".datasets/asymmetric.pa", "wb") as f:
        f.write(response.content)
        
        
if 0:
    
    response = requests.get("https://cs.uef.fi/sipu/datasets/s1.txt")
    with open(".datasets/s1.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/s2.txt")
    with open(".datasets/s2.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/s3.txt")
    with open(".datasets/s3.txt", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/s4.txt")
    with open(".datasets/s4.txt", "wb") as f:
        f.write(response.content)
    
if 0:
    
    response = requests.get("https://cs.uef.fi/sipu/datasets/Aggregation.txt")
    with open(".datasets/Aggregation.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/Compound.txt")
    with open(".datasets/Compound.txt", "wb") as f:
        f.write(response.content)

    response = requests.get("https://cs.uef.fi/sipu/datasets/pathbased.txt")
    with open(".datasets/pathbased.txt", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/spiral.txt")
    with open(".datasets/spiral.txt", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/D31.txt")
    with open(".datasets/D31.txt", "wb") as f:
        f.write(response.content)
        
    response = requests.get("https://cs.uef.fi/sipu/datasets/R15.txt")
    with open(".datasets/R15.txt", "wb") as f:
        f.write(response.content)
        
        response = requests.get("https://cs.uef.fi/sipu/datasets/jain.txt")
    with open(".datasets/jain.txt", "wb") as f:
        f.write(response.content)

        response = requests.get("https://cs.uef.fi/sipu/datasets/flame.txt")
    with open(".datasets/flame.txt", "wb") as f:
        f.write(response.content)

if 0:
        response = requests.get("https://cs.uef.fi/sipu/datasets/t4.8k.txt")
        with open(".datasets/t4.8k.txt", "wb") as f:
            f.write(response.content)

        response = requests.get("http://cs.uef.fi/sipu/datasets/worms.zip")
        with open(".datasets/worms.zip", "wb") as f:
            f.write(response.content)

In [ ]:
if 0:
    import requests

    def download_file(url, local_filename):
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        return local_filename

    url = "http://205.174.165.80/CICDataset/CIC-IDS-2017"
    local_filename = "CIC-IDS-2017"

    download_file(url, local_filename)


In [ ]:
device =torch.device("cpu")

df = pd.read_csv(".datasets/spiral.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])
y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

In [ ]:
flag_plot_acc = 0
num_repetitions = 1

In [ ]:
if 0:
    device =torch.device("cpu")

    df = pd.read_csv(".datasets/spiral.txt", delimiter="\t", header=None)

    X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
    y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
    y = torch.tensor(y-1, dtype=torch.int64).squeeze()

    feature_dim = X.shape[1]
    num_classes = 1
    num_clusters = 3
    rounds = 1
    num_clients = 3

    batch_size = int(np.ceil(len(y)/(num_clients*10)))
    batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
    batch_size = int(2*np.sqrt(np.ceil(len(y))))

    print("Spiral",
        f"\n Samples:   {X.shape[0]}",
        f"\n Features:  {feature_dim}",
        f"\n classes:   {num_classes}")

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "kappa_n":0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": 256,
        "c_max":  10*batch_size,
        "kappa_features": 0.0,
        "device": device,
        "thr_relevance":1
    }
    federated_model_params = local_model_params

    df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="Spiral",
                debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        out_root=".benchmarks",
    )
    print(df.shape)



In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/s4.txt", delimiter="\s+", header=None, engine='python')
df_y = pd.read_csv(".datasets/s4-label.pa", delimiter=" ", header=None, skiprows=5, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
feature_dim = X.shape[1]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 15
rounds = 1
num_clients = 3
num_classes = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.ceil(len(y)/(feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))
print("s4",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max":  2*batch_size,
    "kappa_features": 0.0,
    "device": device,
    "thr_relevance": 1
}
federated_model_params = local_model_params

federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="S4",
        debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        out_root=".benchmarks",
    )


In [ ]:


device = torch.device("cpu")

df_x = pd.read_csv(".datasets/Unbalance2.txt", delimiter=" ", header=None, engine="python")
df_y = pd.read_csv(".datasets/Unbalance2.pa", delimiter=" ", header=None, skiprows=4, engine="python")

X = df_x.values
y = df_y.values
scaler = MinMaxScaler()
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y - 1, dtype=torch.int64).squeeze() 
num_clusters = 8
feature_dim = X.shape[1]
num_classes = 1
rounds = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print(
    "Unbalance2",
    f"\n Samples:   {X.shape[0]}",
    f"\n Features:  {feature_dim}",
    f"\n classes:   {num_classes}"
)

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r":max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
    "thr_relevance": 1
}
federated_model_params = local_model_params

if 0:
    %load_ext line_profiler
    %lprun -f run_benchmark_experiment \
    -f run_benchmark_experiment \
    -f eFDPC.batch_clustering \
    -f eFDPC.recluster_outliers \
    -f train_model_prealocated \
    -f MathOps.compute_batched_activation_diagonal.__wrapped__ \
    -f ConsequenceOps.batch_recursive_least_squares \
    -f ClusteringOps.compute_fisher_scores \
    -f ClusteringOps.batch_increment_clusters.__wrapped__ \
    -f ClusteringOps.kmeans_plusplus_init \
    -f ClusteringOps.reduce_by_label \
    -f ClusteringOps.batch_update_clusters \
    -f ClusteringOps.add_new_clusters.__wrapped__ \
    -f ClusteringOps.fps_mahalanobis \
    -f ClusteringOps.batch_update_global_statistics \
    -f MergingMechanism.batch_merging_mechanism \
    -f MergingMechanism.batch_compute_volume.__wrapped__ \
    -f MergingMechanism.compute_kappa \
    -f MergingMechanism.find_pairs \
    -f MergingMechanism.perform_merge \
    -f FederalOps.merge_model_privately \
    -f FederalOps.federated_merging \
    -f RemovalMechanism.batch_update_scores \
    -f RemovalMechanism.remove_clusters \
    -f HierarchicalClustering.compute_local_density \
    -f HierarchicalClustering.compute_imp_delta_threshold \
    -f HierarchicalClustering.compute_groups \
    -f HierarchicalClustering.cluster_communities \
    -f HierarchicalClustering._compute_delta_to_higher_on \
    run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="Unbalance", debug_flag=0, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

else:
    df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="Unbalance",
        debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        out_root=".benchmarks",
        save_vec=True,
    )

In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/overlap.txt", delimiter=" ", header=None, engine='python')
df_y = pd.read_csv(".datasets/overlap.pa", delimiter=" ", header=None, skiprows=4, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 6
num_clients = 3
rounds = 1
feature_dim = X.shape[1]
num_classes = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("overlap",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features":1,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params
if 0:
    %load_ext line_profiler
    %lprun -f run_benchmark_experiment \
    -f eFDPC.batch_clustering \
    -f train_model_prealocated \
    -f MathOps.compute_batched_activation_diagonal.__wrapped__ \
    -f ConsequenceOps.batch_recursive_least_squares \
    -f ClusteringOps.compute_fisher_scores \
    -f ClusteringOps.batch_increment_clusters.__wrapped__ \
    -f ClusteringOps.kmeans_plusplus_init \
    -f ClusteringOps.reduce_by_label \
    -f ClusteringOps.batch_update_clusters \
    -f ClusteringOps.add_new_clusters.__wrapped__ \
    -f ClusteringOps.fps_mahalanobis \
    -f ClusteringOps.batch_update_global_statistics \
    -f MergingMechanism.batch_merging_mechanism \
    -f MergingMechanism.batch_compute_volume.__wrapped__ \
    -f MergingMechanism.compute_kappa \
    -f MergingMechanism.find_pairs \
    -f MergingMechanism.perform_merge \
    -f FederalOps.merge_model_privately \
    -f FederalOps.federated_merging \
    -f RemovalMechanism.batch_update_scores \
    -f RemovalMechanism.remove_clusters \
    -f HierarchicalClustering.compute_local_density \
    -f HierarchicalClustering.compute_imp_delta_threshold \
    -f HierarchicalClustering.compute_groups \
    -f HierarchicalClustering.cluster_communities \
    run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="Overlap", debug_flag=10, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

elif 1:
    df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="Overlap",
        debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        out_root=".benchmarks",
        save_vec=True,
    )

In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/skewed.txt", delimiter=" ", header=None, engine='python')
df_y = pd.read_csv(".datasets/skewed.pa", delimiter=" ", header=None, skiprows=4, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()
feature_dim = X.shape[1]
num_classes = 1
num_clients = 3
num_clusters = 6
rounds = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("skewed",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features":1,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, 
                                            debug_flag=0,
                                             flag_noniid=True, name="Skewed",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

 


In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/asymmetric.txt", delimiter=" ", header=None, engine='python')
df_y = pd.read_csv(".datasets/asymmetric.pa", delimiter=" ", header=None, skiprows=4, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 5
rounds = 1
feature_dim = X.shape[1]
num_classes = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("asymmetric",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r":max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features":1,
    "device": device,
     "thr_relevance": 1 
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="Asymmetric",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/s1.txt", delimiter="\s+", header=None, engine='python')
df_y = pd.read_csv(".datasets/s1-label.pa", delimiter=" ", header=None, skiprows=5, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 15
feature_dim = X.shape[1]
num_classes = 1
rounds = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.sqrt(np.ceil(len(y))))
print("S1",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": (feature_dim),
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params
 
if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions,
                                             debug_flag=0, 
                                             flag_noniid=True, name="S1",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/s2.txt", delimiter="\s+", header=None, engine='python')
df_y = pd.read_csv(".datasets/s2-label.pa", delimiter=" ", header=None, skiprows=5, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 15

num_classes = 1
num_mini_batches = 10
rounds = 1
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.sqrt(np.ceil(len(y))))
print("s2",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": (feature_dim),
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, 
                                            num_repetitions=num_repetitions,
                                                debug_flag=0,
                                              flag_noniid=True, name="S2",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

    

In [ ]:
device = torch.device("cpu")

df_x = pd.read_csv(".datasets/s3.txt", delimiter="\s+", header=None, engine='python')
df_y = pd.read_csv(".datasets/s3-label.pa", delimiter=" ", header=None, skiprows=5, engine='python')

X = df_x.values
y = df_y.values 
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)

y = torch.tensor(y-1, dtype=torch.int64).squeeze()

num_clusters = 15
rounds = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.sqrt(np.ceil(len(y))))

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": (feature_dim),
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

print("s3",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions,
                                            debug_flag=0,
                                             flag_noniid=True, name="S3",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:
device = torch.device("cpu")

df = pd.read_csv(".datasets/Aggregation.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])
y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

feature_dim = X.shape[1]
num_classes = 1
rounds = 1
num_clusters = 7
num_clients = 3
batch_size = int(np.ceil(len(y)/(num_clients*10)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("Aggregation",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r":max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, debug_flag= 0, name="Aggregation",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)


In [ ]:

device = torch.device("cpu")

df = pd.read_csv(".datasets/Compound.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])
y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

feature_dim = X.shape[1]
num_classes = 1
num_clusters = 6

rounds = 1
batch_size = int(np.ceil(len(y)/(num_clients*10)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("Compound",
    f"\n Samples:   {X.shape[0]}",
    f"\n Features:  {feature_dim}",
    f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r":max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, 
                                            rounds=rounds, num_repetitions=num_repetitions,
                                            debug_flag= 0,
                                              flag_noniid=True, name="Compound",  flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:

device = torch.device("cpu")

df = pd.read_csv(".datasets/pathbased.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])
y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()
    
feature_dim = X.shape[1]
num_classes = 1
rounds = 1
num_clusters = 3
batch_size = int(np.ceil(len(y)/(num_clients*10)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("Compound",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
     "kappa_n": 0,
    "num_sigma": (feature_dim),
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, 
                                            local_model_params, 
                                            federated_model_params, 
                                            num_clients, batch_size, num_clusters, 
                                            rounds=rounds, num_repetitions=num_repetitions, 
                                            debug_flag=0,
                                            flag_noniid=True, name="Pathbased", 
                                              flag_save_img=False, flag_axis=False, 
                                              out_root=".benchmarks", save_vec=True)


In [ ]:

device = torch.device("cpu")

df = pd.read_csv(".datasets/D31.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])

y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()
num_clusters = len(np.unique(y))

feature_dim = X.shape[1]
num_classes = 1
rounds = 1
batch_size = int(np.ceil(len(y)/(num_clients*10)))
batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("D31",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n": 0,
    "num_sigma": (feature_dim),
    "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max": 2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds,
                                             num_repetitions=num_repetitions, flag_noniid=True, name="D31", 
                                             debug_flag=0, 
                                             flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:

device =torch.device("cpu")

df = pd.read_csv(".datasets/R15.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])

y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

feature_dim = X.shape[1]
num_classes = 1
num_clusters = 15
rounds = 1
batch_size = int(np.ceil(len(y)/(num_clients*10)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("R15",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
     "kappa_n": 0,
    "num_sigma": feature_dim,
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r": max(4* num_clusters, 2 * feature_dim),
    "c_max":2*batch_size,
    "kappa_features": 0.0,
    "device": device,
     "thr_relevance": 1
}
federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="R15", debug_flag=0, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)



In [ ]:

device =torch.device("cpu")

df = pd.read_csv(".datasets/jain.txt", delimiter="\t", header=None)

X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])

y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
y = torch.tensor(y-1, dtype=torch.int64).squeeze()

feature_dim = X.shape[1]
num_classes = 1
num_clusters = 2
rounds = 1
num_mini_batches = 10
batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
batch_size = int(np.ceil(len(y)/(feature_dim)))
batch_size = int(np.sqrt(np.ceil(len(y))))

print("jain",
      f"\n Samples:   {X.shape[0]}",
      f"\n Features:  {feature_dim}",
      f"\n classes:   {num_classes}")

local_model_params = {
    "feature_dim": feature_dim,
    "num_classes": num_classes,
    "kappa_n":0,
    "num_sigma":(feature_dim),
    "kappa_join":float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
    "N_r":max(4* num_clusters, 2 * feature_dim),
    "c_max":2*batch_size,
    "kappa_features": 0.0,
    "device": device,
        "thr_relevance":1}

federated_model_params = local_model_params

if 1:
    df, csv_path = run_benchmark_experiment(X, y, local_model_params, 
                                            federated_model_params, num_clients,
                                              batch_size, num_clusters, rounds=rounds, 
                                              num_repetitions=num_repetitions, 
                                              flag_noniid=True, name="Jain",
                                              debug_flag=0,
                                                 flag_save_img=False,
                                                  flag_axis=False, out_root=".benchmarks", 
                                                  save_vec=True)



In [ ]:
if 1:
    device = torch.device("cpu")

    df = pd.read_csv(".datasets/flame.txt", delimiter="\t", header=None)

    X = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,0:2]
    X  = (X  - torch.min(X, dim=0)[0]) / (torch.max(X, dim=0)[0] - torch.min(X, dim=0)[0])

    y = torch.tensor(pd.get_dummies(df).values, dtype=torch.float32)[:,2]
    y = torch.tensor(y-1, dtype=torch.int64).squeeze()
    
    feature_dim = X.shape[1]
    num_classes = 1
    num_clients = 3
    num_clusters = 2
    rounds = 1
    batch_size = int(np.ceil(len(y)/(num_clients*10)))
    batch_size = int(np.ceil(len(y)/(feature_dim)))
    batch_size = int(np.sqrt(np.ceil(len(y))))      
    print("flame",
        f"\n Samples:   {X.shape[0]}",
        f"\n Features:  {feature_dim}",
        f"\n classes:   {num_classes}")

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r":max(4* num_clusters, 2 * feature_dim),
        "c_max": 2*batch_size,
        "kappa_features": 0.0,
        "device": device,
        "thr_relevance": 1
    }
    federated_model_params = local_model_params

    if 1:
        df, csv_path = run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters,
                                                 rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="Flame",
                                                   debug_flag=0, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)


In [ ]:
from pathlib import Path
import io, zipfile, tarfile
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import load_iris as sk_load_iris
from sklearn.datasets import fetch_openml

DATA_DIR = Path(".datasets/uci_suite")

def download(url: str, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    import urllib.request
    urllib.request.urlretrieve(url, out_path)
    return out_path

def to_int_labels(y):
    return pd.Series(y).astype("category").cat.codes.to_numpy(dtype=np.int64)

def minmax_to_torch(X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X.astype(np.float64))
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.int64).squeeze()

def load_openml_sklearn(data_id: int):
    X, y = fetch_openml(data_id=data_id, as_frame=False, return_X_y=True, parser="auto")
    y = to_int_labels(y)
    return X, y

def load_wine():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
    p = download(url, DATA_DIR / "wine" / "wine.data")
    df = pd.read_csv(p, header=None)
    y = df.iloc[:, 0].to_numpy() - 1
    X = df.iloc[:, 1:].to_numpy()
    return X, y

def load_seeds():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt"
    p = download(url, DATA_DIR / "seeds" / "seeds_dataset.txt")
    df = pd.read_csv(p, header=None, sep=r"\s+", engine="python")
    y = df.iloc[:, -1].to_numpy() - 1
    X = df.iloc[:, :-1].to_numpy()
    return X, y

def load_libras():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/libras/movement_libras.data"
    p = download(url, DATA_DIR / "libras" / "movement_libras.data")
    df = pd.read_csv(p, header=None)
    y = df.iloc[:, -1].to_numpy() - 1
    X = df.iloc[:, :-1].to_numpy()
    return X, y

def load_dermatology():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data"
    p = download(url, DATA_DIR / "dermatology" / "dermatology.data")
    df = pd.read_csv(p, header=None).replace("?", np.nan).dropna()
    df = df.apply(pd.to_numeric)
    y = df.iloc[:, -1].to_numpy() - 1
    X = df.iloc[:, :-1].to_numpy()
    return X, y

def load_wdbc():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
    p = download(url, DATA_DIR / "wdbc" / "wdbc.data")
    df = pd.read_csv(p, header=None)
    y = df.iloc[:, 1].map({"B": 0, "M": 1}).to_numpy()
    X = df.iloc[:, 2:].to_numpy()
    return X, y

def load_glass():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data"
    p = download(url, DATA_DIR / "glass" / "glass.data")
    df = pd.read_csv(p, header=None)
    X = df.iloc[:, 1:-1].to_numpy()
    y = to_int_labels(df.iloc[:, -1].to_numpy())
    return X, y

def load_heart_statlog():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/heart/heart.dat"
    p = download(url, DATA_DIR / "heart" / "heart.dat")
    df = pd.read_csv(p, header=None, sep=r"\s+", engine="python")
    X = df.iloc[:, :-1].to_numpy()
    y = to_int_labels(df.iloc[:, -1].to_numpy())
    return X, y

def load_ionosphere():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data"
    p = download(url, DATA_DIR / "ionosphere" / "ionosphere.data")
    df = pd.read_csv(p, header=None)
    X = df.iloc[:, :-1].to_numpy()
    y = to_int_labels(df.iloc[:, -1].to_numpy())
    return X, y

def load_iris():
    X, y = sk_load_iris(return_X_y=True)
    return X, y

def load_new_thyroid():
    url = "http://www.keel.es/dataset/uci/new-thyroid.data"
    p = download(url, DATA_DIR / "thyroid" / "new-thyroid.data")
    df = pd.read_csv(p, header=None, sep=r"[,\s]+", engine="python")
    y = to_int_labels(df.iloc[:, 0].to_numpy())
    X = df.iloc[:, 1:].to_numpy()
    return X, y

def load_waveform():
    return load_openml_sklearn(60)

def load_texture():
    return load_openml_sklearn(40499)

def load_pima():
    return load_openml_sklearn(37)

def load_leaf():
    url = "https://archive.ics.uci.edu/static/public/288/leaf.zip"
    zpath = download(url, DATA_DIR / "leaf" / "leaf.zip")

    with zipfile.ZipFile(zpath, "r") as zf:
        cand = []
        for n in zf.namelist():
            nl = n.lower()
            if nl.endswith((".csv", ".data", ".txt")) and ("image" not in nl) and ("png" not in nl) and ("jpg" not in nl):
                cand.append(n)

        if not cand:
            raise RuntimeError("Leaf zip downloaded but no obvious tabular file found inside. Inspect zf.namelist().")

        cand.sort(key=lambda n: zf.getinfo(n).file_size)
        name = cand[0]

        raw = zf.read(name)
        df = pd.read_csv(io.BytesIO(raw), header=None, sep=None, engine="python")

    col0_u = df.iloc[:, 0].nunique(dropna=True)
    last_u = df.iloc[:, -1].nunique(dropna=True)

    if col0_u <= 40 and last_u > col0_u:
        y = to_int_labels(df.iloc[:, 0].to_numpy())
        X = df.iloc[:, 1:].to_numpy()
    else:
        y = to_int_labels(df.iloc[:, -1].to_numpy())
        X = df.iloc[:, :-1].to_numpy()

    return X, y

def load_titanic():
    url = "https://ftp.cs.toronto.edu/pub/neuron/delve/data/tarfiles/titanic.tar.gz"
    tpath = download(url, DATA_DIR / "titanic" / "titanic.tar.gz")

    with tarfile.open(tpath, "r:gz") as tf:
        members = [m for m in tf.getmembers() if m.isfile()]
        cand = [m for m in members if m.name.lower().endswith((".dat", ".data", ".txt"))]
        if not cand:
            raise RuntimeError("Titanic tar.gz downloaded but no .dat/.data/.txt found inside.")
        cand.sort(key=lambda m: m.size)
        m = cand[0]
        raw = tf.extractfile(m).read().decode("utf-8", errors="ignore")

    rows = []
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith(("#","%","@")):
            continue
        parts = line.split()
        if len(parts) < 4:
            continue
        rows.append(parts[:4])

    df = pd.DataFrame(rows, columns=["class","age","sex","survived"])
    for c in ["class","age","sex","survived"]:
        df[c] = pd.Categorical(df[c]).codes

    X = df[["class","age","sex"]].to_numpy()
    y = to_int_labels(df["survived"].to_numpy())
    return X, y

LOADERS = {
        "Iris": load_iris,
    "Titanic": load_titanic,
    "Waveform": load_waveform,
    "Texture": load_texture,
    "Leaf": load_leaf,
    "Wine": load_wine,
    "Seeds": load_seeds,
    "Libras": load_libras,
    "Dermatology": load_dermatology,
    "WDBC": load_wdbc,
    "Glass": load_glass,
    "Heart": load_heart_statlog,
    "Ionosphere": load_ionosphere,
    "Pima": load_pima,
    "Thyroid": load_new_thyroid,

}
NR_BY_DATASET = {}
{
    "Wine": 16,
    "Seeds": 16,
    "Libras": 128,
    "Dermatology": 8,
    "WDBC": 128,
    "Glass": 32 ,
    "Heart": 16,
    "Ionosphere": 16 ,
    "Iris": 32,
    "Pima": 16  ,
    "Thyroid": 8,
    "Waveform": 64  ,
    "Texture": 256,
    "Leaf": 64,
    "Titanic": 16  ,
}



In [ ]:

import numpy as np
import pandas as pd
import torch

device = torch.device("cpu")

num_clients = 3
rounds = 1
debug_flag = 1
flag_noniid = True
out_root = ".benchmarks"
save_vec = True
flag_save_img = False
flag_axis = False

DEFAULT_N_R = 16
DEFAULT_KAPPA_JOIN = 1.5

def run_ds(
    DATASET_NAME: str,
    Nr_override: int | None = None,
    kappa_join_override: float | None = None,
    run_tag: str | None = None,
    overwrite_dataset_csv: bool = True,
):

    X_np, y_np = LOADERS[DATASET_NAME]()
    X, y = minmax_to_torch(X_np, y_np)
    y = y.squeeze().long()

    feature_dim  = int(X.shape[1])
    num_classes  = 1
    num_clusters = int(torch.unique(y).numel())
    batch_size   = int(np.sqrt(int(len(y))))

    DEFAULT_N_R = max(4* num_clusters, feature_dim)
    N_r = DEFAULT_N_R
    kappa_join = float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim)

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(kappa_join),
        "N_r": N_r,
        "c_max": 2*batch_size,
        "kappa_features": 0.0,
        "device": device,
        "thr_relevance": 1,
    }
    federated_model_params = local_model_params

    run_name = f"{DATASET_NAME}"
    
    df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=flag_noniid,
        name=run_name,
        dataset_key=DATASET_NAME,
        debug_flag=debug_flag,
        flag_save_img=flag_save_img,
        out_root=out_root,
        save_vec=save_vec,
        overwrite_dataset_csv=overwrite_dataset_csv,
    )

    df = df.copy()
    df["N_r"] = N_r
    df["kappa_join"] = kappa_join
    return df, csv_path

if 0:
    flag_plot_acc = 0
    import sys, io, traceback
    from contextlib import contextmanager
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    num_repetitions = 20

    @contextmanager
    def mute_output(mute=True):
        if not mute:
            yield
            return
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = io.StringIO(), io.StringIO()
        try:
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

    def _find_rep_col(df: pd.DataFrame) -> str | None:
        for c in ["rep", "repetition", "repeat", "trial", "run", "seed", "rep_idx"]:
            if c in df.columns:
                return c
        return None

    def _collect_rep_acc(df: pd.DataFrame, *, ds: str, Nr: int, kj: float, method_prefix="Our_eFDPC"):
        if df is None or "method" not in df.columns or "ACC" not in df.columns:
            return []

        ef = df[df["method"].astype(str).str.startswith(method_prefix)].copy()
        if ef.empty:
            return []

        ef["ACC"] = pd.to_numeric(ef["ACC"], errors="coerce")
        rep_col = _find_rep_col(ef)

        out = []
        for m in ef["method"].astype(str).unique():
            sub = ef[ef["method"].astype(str) == m].copy()

            if rep_col is not None:
                acc_by_rep = sub.groupby(rep_col, as_index=False)["ACC"].mean()
                for _, r in acc_by_rep.iterrows():
                    acc = float(r["ACC"])
                    if np.isfinite(acc):
                        out.append(dict(dataset=ds, method=m, Nr=int(Nr), kappa_join=float(kj),
                                        rep=int(r[rep_col]), ACC=acc))
            else:
                for i, acc in enumerate(sub["ACC"].to_numpy()):
                    if np.isfinite(acc):
                        out.append(dict(dataset=ds, method=m, Nr=int(Nr), kappa_join=float(kj),
                                        rep=int(i), ACC=float(acc)))
        return out

    def _summarize(raw: pd.DataFrame) -> pd.DataFrame:
        s = (
            raw.groupby(["dataset", "method", "Nr", "kappa_join"], as_index=False)["ACC"]
            .agg(mean="mean", std="std", n="count")
        )
        s["std"] = s["std"].fillna(0.0)
        return s.sort_values(["dataset", "method", "kappa_join", "Nr"]).reset_index(drop=True)

    def _paper_rc():
        plt.rcParams.update({
            "font.size": 9,
            "axes.labelsize": 9,
            "legend.fontsize": 7,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "lines.linewidth": 1.0,
            "lines.markersize": 4,
            "figure.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.01,
        })

    def _methods_to_plot(ds_sum: pd.DataFrame):
        methods = sorted(ds_sum["method"].unique())
        group = [m for m in methods if "group" in m.lower()]
        return group if group else methods

    def _best_rows(ds_sum: pd.DataFrame, *, ds: str, methods: list[str]) -> pd.DataFrame:
        out = []
        for m in methods:
            sub = ds_sum[(ds_sum["dataset"] == ds) & (ds_sum["method"] == m)].copy()
            if sub.empty:
                continue
            i = int(sub["mean"].to_numpy().argmax())
            r = sub.iloc[i]
            out.append(dict(dataset=ds, method=m, Nr=int(r["Nr"]), kappa_join=float(r["kappa_join"]),
                            mean=float(r["mean"]), std=float(r["std"]), n=int(r["n"])))
        return pd.DataFrame(out).sort_values(["dataset", "method"]).reset_index(drop=True)

    SCALE = 0.7
    BASE_W, BASE_H = 3.35, 2.1
    FIG_W, FIG_H = BASE_W * SCALE, BASE_H * SCALE

    def _plot_acc_vs_nr(ds_sum: pd.DataFrame, *, ds: str, method: str, out_dir: Path, show: bool = True):
        _paper_rc()
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

        subm = ds_sum[(ds_sum["dataset"] == ds) & (ds_sum["method"] == method)].copy()
        for kj in sorted(subm["kappa_join"].unique()):
            sub = subm[subm["kappa_join"] == kj].sort_values("Nr")
            if sub.empty:
                continue
            ax.errorbar(
                sub["Nr"].to_numpy(),
                sub["mean"].to_numpy(),
                yerr=sub["std"].to_numpy(),
                marker="o",
                linestyle="-",
                capsize=2,
                label=rf"$\kappa_{{\mathrm{{join}}}}={kj:g}$",
            )

        ax.set_xlabel(r"$N_r$")
        ax.set_ylabel("ACC")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, loc="best")
        fig.tight_layout()

        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f"{ds}__{method}__ACC_vs_Nr.pdf")
        fig.savefig(out_dir / f"{ds}__{method}__ACC_vs_Nr.svg")

        if show:
            display(fig)
        plt.close(fig)

    def _plot_acc_vs_kj(ds_sum: pd.DataFrame, *, ds: str, method: str, out_dir: Path, show: bool = True):
        _paper_rc()
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

        subm = ds_sum[(ds_sum["dataset"] == ds) & (ds_sum["method"] == method)].copy()
        for Nr in sorted(subm["Nr"].unique()):
            sub = subm[subm["Nr"] == Nr].sort_values("kappa_join")
            if sub.empty:
                continue
            ax.errorbar(
                sub["kappa_join"].to_numpy(),
                sub["mean"].to_numpy(),
                yerr=sub["std"].to_numpy(),
                marker="o",
                linestyle="-",
                capsize=2,
                label=rf"$N_r={int(Nr)}$",
            )

        ax.set_xlabel(r"$\kappa_{\mathrm{join}}$")
        ax.set_ylabel("ACC")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, loc="best")
        fig.tight_layout()

        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f"{ds}__{method}__ACC_vs_kappa_join.pdf")
        fig.savefig(out_dir / f"{ds}__{method}__ACC_vs_kappa_join.svg")

        if show:
            display(fig)
        plt.close(fig)

    Nr_list = [8, 16, 32]
    kappa_join_list = [round(1.5 + 0 * i, 2) for i in range(0, 1)]
    DATASET_LIST = list(LOADERS.keys())

    sweep_dir = Path(out_root) / "sweep_tables"
    plot_dir  = Path(out_root) / "sweep_plots_paper"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    all_records = []

    for ds in DATASET_LIST:
        ds_records = []
        first_combo = True

        for Nr in Nr_list:
            for kj in kappa_join_list:
                print(f"{ds} | Nr={Nr} | kappa_join={kj}", flush=True)
                try:
                    with mute_output(True):
                        df, _ = run_ds(
                            ds,
                            Nr_override=Nr,
                            kappa_join_override=kj,
                            overwrite_dataset_csv=first_combo,
                        )
                    first_combo = False
                except Exception as e:
                    print(f"\nFAIL: ds={ds} Nr={Nr} kj={kj}  {type(e).__name__}: {e}")
                    traceback.print_exc()
                    print()
                    continue

                combo_recs = _collect_rep_acc(df, ds=ds, Nr=Nr, kj=kj, method_prefix="Our_eFDPC")
                if combo_recs:
                    ds_records.extend(combo_recs)
                    all_records.extend(combo_recs)

        if not ds_records:
            print(f"\n{ds}: no Our_eFDPC rows\n")
            continue

        ds_raw = pd.DataFrame(ds_records)
        ds_sum = _summarize(ds_raw)

        ds_raw.to_csv(sweep_dir / f"{ds}_raw.csv", index=False)
        ds_sum.to_csv(sweep_dir / f"{ds}_summary.csv", index=False)

        methods = _methods_to_plot(ds_sum)
        best_df = _best_rows(ds_sum, ds=ds, methods=methods)
        if not best_df.empty:
            print(f"\nBEST (by mean ACC) | {ds}")
            for _, r in best_df.iterrows():
                print(f"  {r['method']} : Nr={int(r['Nr'])}  kappa_join={float(r['kappa_join']):g}  "
                      f"ACC={float(r['mean']):.6f} ± {float(r['std']):.6f}  (n={int(r['n'])})")
            display(best_df)

        for method in methods:
            _plot_acc_vs_nr(ds_sum, ds=ds, method=method, out_dir=plot_dir, show=True)
            _plot_acc_vs_kj(ds_sum, ds=ds, method=method, out_dir=plot_dir, show=True)

    def _pick_method(all_sum: pd.DataFrame) -> str:
        methods = [m for m in all_sum["method"].unique() if str(m).startswith("Our_eFDPC")]
        if not methods:
            return ""
        for m in methods:
            if "group" in str(m).lower():
                return str(m)
        return str(methods[0])

    def _plot_relative_with_lines(all_records, *, out_root: str):
        if not all_records:
            print("No records to plot.")
            return

        out_dir = Path(out_root) / "sweep_plots_paper"
        out_dir.mkdir(parents=True, exist_ok=True)

        raw = pd.DataFrame(all_records)

        summ = (
            raw.groupby(["dataset", "method", "Nr", "kappa_join"], as_index=False)["ACC"]
            .agg(mean="mean", std="std", n="count")
        )
        summ["std"] = summ["std"].fillna(0.0)

        method = _pick_method(summ)
        if method == "":
            print("No Our_eFDPC methods found.")
            return

        dfm = summ[summ["method"].astype(str) == method].copy()
        if dfm.empty:
            print(f"No rows for method={method}.")
            return

        max_by_ds = dfm.groupby("dataset")["mean"].max()
        dfm["max_ds"] = dfm["dataset"].map(max_by_ds)
        dfm["rel"] = dfm["mean"] / dfm["max_ds"].replace(0, np.nan)
        dfm = dfm[np.isfinite(dfm["rel"])].copy()

        grid = (
            dfm.groupby(["Nr", "kappa_join"], as_index=False)["rel"]
            .mean()
            .rename(columns={"rel": "rel_mean"})
        )

        i = int(grid["rel_mean"].to_numpy().argmax())
        br = grid.iloc[i]
        print("\nBEST (global, avg relative over datasets)")
        print(f"  method={method} : Nr={int(br['Nr'])}  kappa_join={float(br['kappa_join']):g}  relACC={float(br['rel_mean']):.6f}")
        display(grid.sort_values(["rel_mean"], ascending=False).head(10))

        nrs = sorted(grid["Nr"].unique())
        kjs = sorted(grid["kappa_join"].unique())

        _paper_rc()

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        for kj in kjs:
            sub = grid[grid["kappa_join"] == kj].sort_values("Nr")
            if sub.empty:
                continue
            ax.plot(sub["Nr"].to_numpy(), sub["rel_mean"].to_numpy(), marker="o", linestyle="-",
                    label=rf"$\kappa_{{\mathrm{{join}}}}={kj:g}$")

        ax.set_xlabel(r"$N_r$")
        ax.set_ylabel("Relative ACC")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, loc="best")
        fig.tight_layout()
        fig.savefig(out_dir / f"_ALL__{method}__relACC_vs_Nr__lines_kj.pdf")
        fig.savefig(out_dir / f"_ALL__{method}__relACC_vs_Nr__lines_kj.svg")
        display(fig)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        for Nr in nrs:
            sub = grid[grid["Nr"] == Nr].sort_values("kappa_join")
            if sub.empty:
                continue
            ax.plot(sub["kappa_join"].to_numpy(), sub["rel_mean"].to_numpy(), marker="o", linestyle="-",
                    label=rf"$N_r={int(Nr)}$")

        ax.set_xlabel(r"$\kappa_{\mathrm{join}}$")
        ax.set_ylabel("Relative ACC")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, loc="best")
        fig.tight_layout()
        fig.savefig(out_dir / f"_ALL__{method}__relACC_vs_kappa_join__lines_Nr.pdf")
        fig.savefig(out_dir / f"_ALL__{method}__relACC_vs_kappa_join__lines_Nr.svg")
        display(fig)
        plt.close(fig)

        print(f"Saved in: {out_dir}")
        print(f"Method used: {method}")

    _plot_relative_with_lines(all_records, out_root=out_root)

In [ ]:
df_Thyroid , Thyroid_csv = run_ds("Thyroid")

In [ ]:
df_Libras, Libras_csv = run_ds("Libras")

In [ ]:
df_Dermatology, Dermatology_csv = run_ds("Dermatology")

In [ ]:
df_Waveform, Waveform_csv = run_ds("Waveform")

In [ ]:
df_Ionosphere, Ionosphere_csv = run_ds("Ionosphere")

In [ ]:
df_iris, iris_csv = run_ds("Iris")

In [ ]:
df_wine, wine_csv = run_ds("Wine")
wine_csv

In [ ]:
df_seeds, seeds_csv = run_ds("Seeds")


In [ ]:
df_wdbc, wdbc_csv = run_ds("WDBC")

In [ ]:
df_glass, glass_csv = run_ds("Glass")   

In [ ]:
df_heart, heart_csv = run_ds("Heart")


In [ ]:
df_pima, pima_csv = run_ds("Pima")

In [ ]:
df_texture, texture_csv = run_ds("Texture")

In [ ]:
df_leaf, leaf_csv = run_ds("Leaf")
        

In [ ]:
import os

image_dir = '.Images/related_methods'

files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

datasets = sorted(set(f.split('_')[0] for f in files))
methods = sorted(set(f.split('_', 1)[1].rsplit('.', 1)[0] for f in files if len(f.split('_')) > 1))

print("Datasets:")
for dataset in datasets:
    print(dataset)

print("\nMethods:")
for method in methods:
    print(method)


In [ ]:
if 0:
        
    import os
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec

    image_dir = '.Images/related_methods/noniid/'

    files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

    datasets_group1 = ['D31', 'R15', 'S1', 'S3', 'Skewed','Unbalance2','Overlap','Aggregation']
    datasets_group2 = ['Aggregation', 'Jain', 'Pathbased', 'Spiral']

    methods = sorted(set(f.split('_', 1)[1].rsplit('.', 1)[0].replace('_', '-') for f in files if len(f.split('_')) > 1))

    if 'eFDPC' in methods:
        methods.remove('eFDPC')
    methods.append('eFDPC (Our)')

    def create_figure(datasets, output_file):
        fig = plt.figure(figsize=(len(methods)*2.5, len(datasets)*2.5))

        gs = GridSpec(len(datasets)+1, len(methods)+1, figure=fig,
                    width_ratios=[0.15] + [1] * len(methods),
                    height_ratios=[0.15] + [1] * len(datasets))

        for j, method in enumerate(methods):
            ax = fig.add_subplot(gs[0, j+1])
            ax.text(0.5, 0.2, method, ha='center', va='center', fontsize=14, fontweight='bold')
            ax.axis('off')

        for i, dataset in enumerate(datasets):
            ax = fig.add_subplot(gs[i+1, 0])
            ax.text(0.2, 0.5, dataset, ha='center', va='center', fontsize=14, fontweight='bold', rotation=90)
            ax.axis('off')

        for i, dataset in enumerate(datasets):
            for j, method in enumerate(methods):
                ax = fig.add_subplot(gs[i+1, j+1])
                image_path = os.path.join(image_dir, f"{dataset}_{method.replace(' (Our)', '').replace('-', '_')}.png")
                if os.path.exists(image_path):
                    img = plt.imread(image_path)
                    ax.imshow(img)
                ax.axis('off')
        
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=-0.1, hspace=0.0)

        plt.savefig(output_file, format='png', bbox_inches='tight', dpi=600)

        plt.show()

    create_figure(datasets_group1, ".Images/final_comparison_group1.png")

    create_figure(datasets_group2, ".Images/final_comparison_group2.png")


In [ ]:
from torchvision import datasets, transforms
import torch
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

if 0:
    mnist_train = datasets.MNIST(root='.datasets', train=True, download=True, transform=transforms.ToTensor())
    mnist_test = datasets.MNIST(root='.datasets', train=False, download=True, transform=transforms.ToTensor())

    transform = transforms.ToTensor()
    mnist_train = datasets.MNIST(root='.datasets', train=True, download=True, transform=transform)

    data_sample_size = 20000
    images, labels = [], []

    for i in range(data_sample_size):
        image, label = mnist_train[i]
        images.append(image.view(-1).numpy())
        labels.append(label)

    images = torch.tensor(images).numpy()
    labels = torch.tensor(labels).numpy()

    pca = PCA(n_components=2)
    images_pca = pca.fit_transform(images)

    plt.figure(figsize=(8, 8))
    scatter = plt.scatter(images_pca[:, 0], images_pca[:, 1], c=labels, cmap='tab10', alpha=0.7, s=15)
    plt.colorbar(scatter)
    plt.title('PCA of MNIST Dataset')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.show()

from torchvision import datasets, transforms
import torch

dataset_path = '.datasets/MNIST/raw'

mnist_train = datasets.MNIST(root='.datasets', train=True, download=not os.path.exists(dataset_path), transform= transforms.ToTensor())

X = torch.stack([mnist_train[i][0].view(-1) for i in range(len(mnist_train))])
y = torch.tensor([mnist_train[i][1] for i in range(len(mnist_train))])

print(X.shape, y.shape)

feature_dim = 784
pca = PCA(n_components=feature_dim)
X_pca = pca.fit_transform(X.numpy())

if 1:
    X = torch.tensor(X_pca)

print(X.shape, y.shape)
y = torch.tensor(y, dtype=torch.int64).squeeze()


In [ ]:

if 0:

    feature_dim = X.shape[1]       
   
    device =  torch.device("cpu")
    num_classes = 1
    num_clients = 3
    rounds = 1
    num_clusters = 10
    num_mini_batches = 10
    batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
    batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
    batch_size = int(np.sqrt(np.ceil(len(y))))
    
    print("MNIST",
        f"\n Samples:   {X.shape[0]}",
        f"\n Features:  {feature_dim}",
        f"\n Clients:   {num_clients}")

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "kappa_n":0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": feature_dim,
        "c_max": 2*batch_size,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1
    }

    federated_model_params = local_model_params
    if 0:
        %load_ext line_profiler
        %lprun -f run_benchmark_experiment \
        -f run_benchmark_experiment \
        -f eFDPC.batch_clustering \
        -f eFDPC.recluster_outliers \
        -f train_model_prealocated \
        -f MathOps.compute_batched_activation_diagonal.__wrapped__ \
        -f ConsequenceOps.batch_recursive_least_squares \
        -f ClusteringOps.batch_increment_clusters.__wrapped__ \
        -f ClusteringOps.kmeans_plusplus_init \
        -f ClusteringOps.reduce_by_label \
        -f ClusteringOps.batch_update_clusters \
        -f ClusteringOps.add_new_clusters.__wrapped__ \
        -f ClusteringOps.fps_mahalanobis \
        -f ClusteringOps.batch_update_global_statistics \
        -f MergingMechanism.batch_merging_mechanism \
        -f MergingMechanism.batch_compute_volume.__wrapped__ \
        -f MergingMechanism.compute_kappa \
        -f MergingMechanism.perform_merge \
        -f FederalOps.merge_model_privately.__wrapped__ \
        -f FederalOps.federated_merging \
        -f RemovalMechanism.batch_update_scores \
        -f RemovalMechanism.remove_clusters \
        -f HierarchicalClustering.compute_local_density.__wrapped__ \
        -f HierarchicalClustering.compute_imp_delta_threshold \
        -f HierarchicalClustering.compute_groups \
        -f HierarchicalClustering.cluster_communities \
        -f HierarchicalClustering._compute_delta_to_higher_on.__wrapped__ \
        run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="MNIST", debug_flag=0, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

        
    else:
           df, csv_path = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="MNIST",
        debug_flag=0,
        out_root=".benchmarks",
        flag_save_img=False,
        flag_axis=False,
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def sweep_clients_all_methods(
    X, y,
    *,
    dataset_id: str,
    exp_id: str = "clientsweep",
    client_grid=(100),
    flag_noniid: bool = True,
    rounds: int = 1,
    num_clusters: int = 10,
    num_repetitions: int = 5,
    local_model_params=None,
    federated_model_params=None,
    out_root: str = ".benchmarks",
    debug_flag: int = 0,
    flag_save_img: bool = False,
    flag_axis: bool = False,
    save_vec: bool = True,
    overwrite_dataset_csv: bool = True,
    dataset_key_per_K: bool = True,
):
    out_root = Path(out_root)
    sweep_dir = out_root / "sweeps"
    sweep_dir.mkdir(parents=True, exist_ok=True)

    split = "noniid" if flag_noniid else "iid"
    N = int(len(y))

    all_dfs = []
    perK_csvs = []

    for K in client_grid:
        K = int(K)

        run_name = f"{exp_id}_{dataset_id}_C{K}_{split}"
        dataset_key = f"{dataset_id}_C{K}" if dataset_key_per_K else dataset_id

        batch_size = int(np.sqrt(N))

        df, csv_path = run_benchmark_experiment(
            X, y,
            local_model_params=local_model_params,
            federated_model_params=federated_model_params,
            num_clients=K,
            batch_size=int(batch_size),
            num_clusters=int(num_clusters),
            rounds=int(rounds),
            num_repetitions=int(num_repetitions),
            flag_noniid=bool(flag_noniid),
            name=run_name,
            debug_flag=int(debug_flag),
            out_root=str(out_root),
            flag_save_img=bool(flag_save_img),
            flag_axis=bool(flag_axis),
            save_vec=bool(save_vec),
            overwrite_dataset_csv=bool(overwrite_dataset_csv),
            dataset_key=dataset_key,
            alpha = 0.9
        )

        df = df.copy()
        df["dataset_id"] = dataset_id
        df["exp_id"] = exp_id
        df["clients"] = K
        df["split"] = split
        df["batch_size"] = int(batch_size)

        all_dfs.append(df)
        perK_csvs.append(str(csv_path))

    sweep_df = pd.concat(all_dfs, ignore_index=True)
    sweep_csv = sweep_dir / f"{exp_id}_{dataset_id}_{split}_ALL.csv"
    sweep_df.to_csv(sweep_csv, index=False)

    df_last = sweep_df
    if "round_idx" in df_last.columns:
        r = pd.to_numeric(df_last["round_idx"], errors="coerce")
        df_last = df_last[r == int(rounds)].copy()

    metric_candidates = ["ACC", "ARI", "NMI", "Silhouette", "runtime", "time", "seconds", "time_sec"]
    metrics = [c for c in metric_candidates if c in df_last.columns]
    for c in metrics:
        df_last[c] = pd.to_numeric(df_last[c], errors="coerce")

    summary_df = (
        df_last.groupby(["dataset_id", "split", "clients", "method"], dropna=False)[metrics]
        .agg(["mean", lambda s: float(np.nanstd(s.to_numpy(dtype=float), ddof=0))])
    )
    summary_df.columns = [f"{m}_{'mean' if stat=='mean' else 'std'}" for (m, stat) in summary_df.columns]
    summary_df = summary_df.reset_index().sort_values(["clients", "method"]).reset_index(drop=True)

    summary_csv = sweep_dir / f"{exp_id}_{dataset_id}_{split}_SUMMARY.csv"
    summary_df.to_csv(summary_csv, index=False)

    return sweep_df, sweep_csv, summary_df, summary_csv, perK_csvs

In [ ]:
if 0:
    dataset_id = "MNIST"

    feature_dim = int(X.shape[1])
    device = torch.device("cpu")
    num_clusters = 10

    flag_plot_acc = 1

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": 1,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": max(4 * num_clusters, feature_dim),
        "c_max": 2 * batch_size,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1,
    }
    federated_model_params = dict(local_model_params)

    sweep_df, sweep_csv, summary_df, summary_csv, perK_csvs = sweep_clients_all_methods(
        X, y,
        dataset_id=dataset_id,
        exp_id="clientsweep",
        client_grid=[5, 10, 30],
        flag_noniid=True,
        rounds=1,
        num_clusters=num_clusters,
        num_repetitions=20,
        local_model_params=local_model_params,
        federated_model_params=federated_model_params,
        out_root=".benchmarks",
        debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        save_vec=True,
        overwrite_dataset_csv=True,
        dataset_key_per_K=True,
    )

    print(sweep_csv)
    print(summary_csv)
    display(summary_df)

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

def sweep_alpha_all_methods(
    X, y,
    *,
    dataset_id: str,
    exp_id: str = "alphasweep",
    alpha_grid=(0.1, 0.3, 0.9, 3.0),
    num_clients: int = 3,
    rounds: int = 1,
    num_clusters: int = 10,
    num_repetitions: int = 20,
    local_model_params=None,
    federated_model_params=None,
    out_root: str = ".benchmarks",
    debug_flag: int = 0,
    flag_save_img: bool = False,
    flag_axis: bool = False,
    save_vec: bool = True,
    overwrite_dataset_csv: bool = True,
    dataset_key_per_alpha: bool = True,
):
    out_root = Path(out_root)
    sweep_dir = out_root / "sweeps"
    sweep_dir.mkdir(parents=True, exist_ok=True)

    split = "noniid"
    N = int(len(y))
    batch_size = int(np.sqrt(N))

    all_dfs = []
    perA_csvs = []

    for a in alpha_grid:
        a = float(a)

        run_name = f"{exp_id}_{dataset_id}_a{a:g}_C{int(num_clients)}_{split}"
        dataset_key = f"{dataset_id}_a{a:g}" if dataset_key_per_alpha else dataset_id

        df, csv_path = run_benchmark_experiment(
            X, y,
            local_model_params=local_model_params,
            federated_model_params=federated_model_params,
            num_clients=int(num_clients),
            batch_size=int(batch_size),
            num_clusters=int(num_clusters),
            rounds=int(rounds),
            num_repetitions=int(num_repetitions),
            flag_noniid=True,
            name=run_name,
            debug_flag=int(debug_flag),
            out_root=str(out_root),
            flag_save_img=bool(flag_save_img),
            flag_axis=bool(flag_axis),
            save_vec=bool(save_vec),
            overwrite_dataset_csv=bool(overwrite_dataset_csv),
            dataset_key=dataset_key,
            alpha=float(a)
        )

        df = df.copy()
        df["dataset_id"] = dataset_id
        df["exp_id"] = exp_id
        df["clients"] = int(num_clients)
        df["alpha"] = float(a)
        df["split"] = split
        df["batch_size"] = int(batch_size)

        all_dfs.append(df)
        perA_csvs.append(str(csv_path))

    sweep_df = pd.concat(all_dfs, ignore_index=True)
    sweep_csv = sweep_dir / f"{exp_id}_{dataset_id}_{split}_ALL.csv"
    sweep_df.to_csv(sweep_csv, index=False)

    df_last = sweep_df
    if "round_idx" in df_last.columns:
        rr = pd.to_numeric(df_last["round_idx"], errors="coerce")
        df_last = df_last[rr == int(rounds)].copy()

    metric_candidates = ["ACC", "ARI", "NMI", "F1", "time_sec"]
    metrics = [c for c in metric_candidates if c in df_last.columns]
    for c in metrics:
        df_last[c] = pd.to_numeric(df_last[c], errors="coerce")

    summary_df = (
        df_last.groupby(["dataset_id", "split", "clients", "alpha", "method"], dropna=False)[metrics]
        .mean()
        .reset_index()
        .sort_values(["alpha", "method"])
        .reset_index(drop=True)
    )

    summary_csv = sweep_dir / f"{exp_id}_{dataset_id}_{split}_SUMMARY.csv"
    summary_df.to_csv(summary_csv, index=False)

    return sweep_df, sweep_csv, summary_df, summary_csv, perA_csvs

In [ ]:
if 0:
    dataset_id = "MNIST"

    feature_dim = int(X.shape[1])
    device = torch.device("cpu")
    num_clusters = 10

    flag_plot_acc = 0

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": 1,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": max(4 * num_clusters, feature_dim),
        "c_max": 2 * batch_size,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1,
    }
    federated_model_params = dict(local_model_params)

    alpha_grid = [0.1, 0.3, 1.0, 3.0]
    sweep_df, sweep_csv, summary_df, summary_csv, perA_csvs = sweep_alpha_all_methods(
        X, y,
        dataset_id=dataset_id,
        exp_id="alphasweep",
        alpha_grid=alpha_grid,
        num_clients=10,
        rounds=1,
        num_clusters=10,
        num_repetitions=20,
        local_model_params=local_model_params,
        federated_model_params=federated_model_params,
        out_root=".benchmarks",
        debug_flag=0,
    )

In [ ]:
import numpy as np, pandas as pd, time, gc, torch
from pathlib import Path

def sweep_batch_efdpc(X, y, *, batch_grid=None, alpha=0.9, num_clients=3,
                      num_clusters=10, rounds=1, num_reps=20,
                      local_model_params=None, base_seed=1,
                      dataset_id="MNIST", out_root=".benchmarks"):
    N = int(len(y)); base = int(np.sqrt(N))
    if batch_grid is None:
        batch_grid = sorted({max(2, base//4), base//2, base, 2*base, 4*base, 8*base})
    d = int(X.shape[1])
    local_model_params = {**local_model_params, "feature_dim": d}

    sweep_dir = Path(out_root)/"sweeps"; sweep_dir.mkdir(parents=True, exist_ok=True)
    all_csv = sweep_dir / f"batchsweep_{dataset_id}_C{int(num_clients)}_noniid_ALL.csv"
    sum_csv = sweep_dir / f"batchsweep_{dataset_id}_C{int(num_clients)}_noniid_SUMMARY.csv"
    cols = ["dataset","split","batch_size","rep","method","ACC","ARI","NMI","time_sec"]

    def _drop_batch(b):
        if all_csv.exists():
            cur = pd.read_csv(all_csv, on_bad_lines="skip")
            if "batch_size" in cur.columns:
                cur = cur[cur["batch_size"].astype("Int64") != int(b)]
            cur.to_csv(all_csv, index=False)
            return True
        return False

    def _refresh_summary():
        if not all_csv.exists(): return
        d_ = pd.read_csv(all_csv, on_bad_lines="skip").drop_duplicates(
            subset=["batch_size","rep","method"], keep="last")
        (d_.groupby(["batch_size","method"])[["ACC","ARI","NMI","time_sec"]]
           .agg(["mean","std"]).reset_index()).to_csv(sum_csv, index=False)

    print(f"grid={batch_grid}  (ONLY these batch sizes get overwritten; the rest stay)", flush=True)
    for bi, b in enumerate(batch_grid):
        b = int(b)
        lp = {**local_model_params, "c_max": 2*b}; fp = dict(lp)
        write_header = not _drop_batch(b)
        for rep in range(num_reps):
            t0 = time.time()
            set_all_seeds(base_seed + rep)
            (train_loaders, train_datasets, test_loader, test_dataset,
             all_data, train_data) = prepare_non_iid_federated_dataset_for_clustering(
                X, y, num_clients=num_clients, alpha=alpha, batch_size=b)
            ms = run_efdpc_experiment(
                rounds=rounds, num_clients=num_clients, local_model_params=lp,
                federated_model_params=fp, train_datasets=train_datasets,
                train_loaders=train_loaders, test_dataset=test_dataset, batch_size=b,
                num_clusters=num_clusters, profiler_flag=False, debug_flag=0, name=f"bs_b{b}_r{rep}")
            ms = ms if isinstance(ms, (list, tuple)) else [ms]
            rows = [{"dataset": dataset_id, "split": "noniid", "batch_size": b, "rep": rep,
                     "method": m.get("method",""), "ACC": m.get("ACC"), "ARI": m.get("ARI"),
                     "NMI": m.get("NMI"), "time_sec": m.get("time_sec")} for m in ms]
            pd.DataFrame(rows, columns=cols).to_csv(all_csv, mode="a", header=write_header, index=False)
            write_header = False
            acc_g = next((m.get("ACC") for m in ms if m.get("method")=="Our_eFDPC_group"), float("nan"))
            print(f"  b={b} [{bi+1}/{len(batch_grid)}] rep {rep+1}/{num_reps}  ACC(group)={acc_g:.3f}  {time.time()-t0:.1f}s", flush=True)
            del train_loaders, train_datasets, test_loader, test_dataset, all_data, train_data, ms, rows
            gc.collect()
        _refresh_summary()

    df = pd.read_csv(all_csv, on_bad_lines="skip").drop_duplicates(
        subset=["batch_size","rep","method"], keep="last")
    _refresh_summary()
    return df, pd.read_csv(sum_csv)


In [ ]:
if 1:
    dataset_id = "MNIST"

    feature_dim = int(X.shape[1])
    device = torch.device("cpu")
    num_clusters = 10

    flag_plot_acc = 0

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": 1,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": max(4*num_clusters, feature_dim),
        "c_max": 2 * 256,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1,
    }

    batch_grid = [32, 64, 128, 256, 512, 1024]
    df_bs, summary_bs = sweep_batch_efdpc(
        X, y,
        dataset_id=dataset_id,
        batch_grid=batch_grid,
        alpha=0.9,
        num_clients=3,
        num_clusters=num_clusters,
        rounds=1,
        num_reps=20,
        local_model_params=local_model_params,
    )
    print(summary_bs)


In [ ]:
import pandas as pd, numpy as np, re
from pathlib import Path

DATASET = "MNIST"
VARIANTS = [("Our_eFDPC_centers", r"\mbox{eFDPC-$\mathcal{P}$}"),
            ("Our_eFDPC_group",   r"\mbox{eFDPC-$\mathcal{G}$}")]
METRICS = [("ACC",      r"ACC [$10^{-2}$]", 100, "max"),
           ("ARI",      r"ARI [$10^{-2}$]", 100, "max"),
           ("NMI",      r"NMI [$10^{-2}$]", 100, "max"),
           ("time_sec", r"Time [$10^{-1}\,\mathrm{s}$]", 10, "min")]
sweep_dir = Path(".benchmarks/sweeps")

files = {}
for f in sorted(sweep_dir.glob(f"batchsweep_{DATASET}_C*_noniid_ALL.csv")):
    m = re.search(r"_C(\d+)_", f.name)
    if m: files[int(m.group(1))] = f
Ks = sorted(files)
assert Ks, f"no batchsweep_{DATASET}_C*_noniid_ALL.csv in {sweep_dir}"

mkeys = [k for k,_,_,_ in METRICS]
agg, all_b = {}, set()
for K, f in files.items():
    raw = pd.read_csv(f, on_bad_lines="skip")
    for k in mkeys: raw[k] = pd.to_numeric(raw[k], errors="coerce")
    for vkey,_ in VARIANTS:
        g = raw[raw["method"]==vkey].groupby("batch_size")[mkeys].agg(["mean","std"])
        g.columns = [f"{a}_{b}" for a,b in g.columns]
        agg[(K,vkey)] = g; all_b.update(int(b) for b in g.index)
batches = sorted(all_b)

cols = [(K, vkey) for K in Ks for vkey,_ in VARIANTS]
best = {(K,vkey,k): (agg[(K,vkey)][f"{k}_mean"].min() if d=="min" else agg[(K,vkey)][f"{k}_mean"].max())
        for K in Ks for vkey,_ in VARIANTS for k,_,_,d in METRICS}

hu = lambda x, s: int(np.floor(float(x)*s + 0.5))
nv = len(VARIANTS)
colspec = "l|l|" + "|".join(["c"*nv]*len(Ks))

lines = [r"\begin{table}[!t]", r"  \centering", r"  \footnotesize",
        r"  \caption{Mini-batch size $B$ and client counts $K$ sweep for $\alpha{=}0.9$.}\label{tab:batch_sweep_clients}",
         r"  \setlength{\tabcolsep}{3pt}",
         r"  \begin{tabular}{"+colspec+r"}", r"    \toprule",
         r"    \multirow{2}{*}{Dataset} & \multirow{2}{*}{Metric} & " +
            " & ".join(r"\multicolumn{%d}{c}{$K=%d$}" % (nv, K) for K in Ks) + r" \\",
         "    " + "".join(r"\cmidrule(lr){%d-%d}" % (3+i*nv, 2+(i+1)*nv) for i in range(len(Ks))),
         r"     & & " + " & ".join(vlab for _ in Ks for _,vlab in VARIANTS) + r" \\",
         r"    \midrule"]
for bi, b in enumerate(batches):
    for ri, (k, label, scale, _) in enumerate(METRICS):
        cells = []
        for (K, vkey) in cols:
            g = agg[(K,vkey)]
            if b in g.index and not pd.isna(g.loc[b, f"{k}_mean"]):
                mu = g.loc[b, f"{k}_mean"]; sd = g.loc[b, f"{k}_std"]
                sd = 0.0 if pd.isna(sd) else sd
                c = f"{hu(mu,scale):02d}$\\pm${hu(sd,scale):02d}"
                if np.isclose(mu, best[(K,vkey,k)]): c = r"\textbf{" + c + "}"
            else:
                c = "--"
            cells.append(c)
        left = (r"\multirow{%d}{*}{\shortstack[l]{{%s}\\$B=%d$}}" % (len(METRICS), DATASET, b)
                if ri == 0 else "")
        lines.append("    " + left + " & " + label + " & " + " & ".join(cells) + r" \\")
    if bi != len(batches) - 1:
        lines.append(r"    \midrule")
lines += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}"]
print("\n".join(lines))


In [ ]:
import matplotlib.pyplot as plt
m = "Our_eFDPC_group"
sub = df_bs[df_bs["method"] == m]
bs  = sorted(sub["batch_size"].unique())
am  = sub.groupby("batch_size")["ACC"].mean().reindex(bs)
asd = sub.groupby("batch_size")["ACC"].std().reindex(bs)
tm  = sub.groupby("batch_size")["time_sec"].mean().reindex(bs)
tsd = sub.groupby("batch_size")["time_sec"].std().reindex(bs)

fig, ax1 = plt.subplots(figsize=(5,4))
ax1.errorbar(bs, am, yerr=asd, marker="o", color="#2b3a67")
ax1.set_xscale("log", base=2); ax1.set_xlabel(r"Batch size $B$"); ax1.set_ylabel("ACC", color="#2b3a67")
ax2 = ax1.twinx()
ax2.errorbar(bs, tm, yerr=tsd, marker="s", linestyle="--", color="#7a7a7a")
ax2.set_ylabel("Runtime (s)", color="#7a7a7a")
fig.tight_layout()
from pathlib import Path; Path(".benchmarks/sweep_plots_paper").mkdir(parents=True, exist_ok=True)
fig.savefig(".benchmarks/sweep_plots_paper/MNIST__eFDPC__ACC_time_vs_batch.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from urllib.request import urlopen
import os

def download_kdd99():
    os.makedirs(".downloads", exist_ok=True)
    file_path = ".downloads/kddcup.data_10_percent.gz"
    
    if not os.path.exists(file_path):
        print("Downloading KDD99 dataset...")
        url = "http://kdd.ics.uci.edu/databases/kddcup99/kddcup.data_10_percent.gz"
        with urlopen(url) as response, open(file_path, "wb") as out_file:
            out_file.write(response.read())
        print("Dataset downloaded and saved to .downloads.")
    else:
        print("Dataset already exists in .downloads. Skipping download.")
    
    return file_path

def load_data_kdd99(feature_dim, max_samples=None):
    file_path = download_kdd99()
    
    column_names = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
                    "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
                    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login",
                    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
                    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
                    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
                    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "class"]
    
    df = pd.read_csv(file_path, header=None, names=column_names)
    

    y = df['class']
    
    y_cleaned = y.str.strip().str.rstrip('.')
    
    y_binary = y_cleaned.apply(lambda label: 0 if label == 'normal' else 1)
    
        
    if max_samples is not None and max_samples < len(df):
        idx = df.sample(n=max_samples).index
        df = df.loc[idx].reset_index(drop=True)
        y_binary = y_binary.loc[idx].reset_index(drop=True)
        
    y_tensor = torch.tensor(y_binary.values, dtype=torch.int64)

    X = df.drop(columns=['class'])
    categorical_cols = ["protocol_type", "service", "flag", "land", "logged_in", "is_host_login", "is_guest_login"]
    numerical_cols = [col for col in X.columns if col not in categorical_cols]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(sparse_output=False), categorical_cols)
        ])
    
    X_transformed = preprocessor.fit_transform(X)

    print(f"Shape of X: {X_transformed.shape[1]}")
    if feature_dim and feature_dim < X_transformed.shape[1]:
        print(f"Applying PCA to reduce dimensions to {feature_dim} components.")
        pca = PCA(n_components=feature_dim)
        X_reduced = pca.fit_transform(X_transformed)
        X_tensor = torch.tensor(X_reduced, dtype=torch.float32)
    else:
        print("Skipping PCA and using all transformed features.")
        X_tensor = torch.tensor(X_transformed, dtype=torch.float32)
    
    return X_tensor, y_tensor


In [ ]:
if 0:
    feature_dim = 121
    X, y = load_data_kdd99(feature_dim, max_samples=600000)

    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)
    X = torch.tensor(X, dtype=torch.float32)

    print("X shape:", X.shape)
    print("y shape:", y.shape)

    rounds = 1
    num_classes = 1
    num_clients = 3
    num_clusters = 10
    num_mini_batches = 10

    batch_size = int(np.ceil(len(y)/(num_clients*num_mini_batches)))
    batch_size = int(np.ceil(len(y)/(num_clients*feature_dim)))
    batch_size = int(np.sqrt(np.ceil(len(y))))

    print("KDD99 Dataset",
        f"\n Samples:   {X.shape[0]}",
        f"\n Features:  {feature_dim}",
        f"\n Clients:   {num_clients}")

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": float(1.0 + 0.25 * chi2.ppf(0.5, feature_dim)/feature_dim),
        "N_r": feature_dim,
        "c_max": 2*batch_size,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1
    }

    federated_model_params = local_model_params

    if 0:
        %load_ext line_profiler
        %lprun -f run_benchmark_experiment \
        -f eFDPC.batch_clustering \
        -f MathOps.compute_batched_activation_diagonal.__wrapped__ \
        -f ConsequenceOps.batch_recursive_least_squares \
        -f ClusteringOps.compute_fisher_scores \
        -f ClusteringOps.batch_increment_clusters.__wrapped__ \
        -f ClusteringOps.kmeans_plusplus_init \
        -f ClusteringOps.reduce_by_label \
        -f ClusteringOps.batch_update_clusters \
        -f ClusteringOps.add_new_clusters.__wrapped__ \
        -f ClusteringOps.fps_mahalanobis \
        -f ClusteringOps.batch_update_global_statistics \
        -f MergingMechanism.batch_merging_mechanism \
        -f MergingMechanism.batch_compute_volume.__wrapped__ \
        -f MergingMechanism.compute_kappa \
        -f MergingMechanism.find_pairs \
        -f MergingMechanism.perform_merge \
        -f FederalOps.merge_model_privately \
        -f FederalOps.federated_merging \
        -f RemovalMechanism.batch_update_scores \
        -f RemovalMechanism.remove_clusters \
        -f HierarchicalClustering.compute_local_density \
        -f HierarchicalClustering.compute_imp_delta_threshold \
        -f HierarchicalClustering.compute_components_global \
        -f HierarchicalClustering.compute_groups \
        -f HierarchicalClustering.cluster_communities \
        run_benchmark_experiment(X, y, local_model_params, federated_model_params, num_clients, batch_size, num_clusters, rounds=rounds, num_repetitions=num_repetitions, flag_noniid=True, name="KDD99", debug_flag= 0, flag_save_img=False, flag_axis=False, out_root=".benchmarks", save_vec=True)

    elif 1:
      df, paths = run_benchmark_experiment(
        X, y,
        local_model_params, federated_model_params,
        num_clients, batch_size, num_clusters,
        rounds=rounds,
        num_repetitions=num_repetitions,
        flag_noniid=True,
        name="KDD99",
        debug_flag=0,
        out_root=".benchmarks",
        flag_save_img=False,
        flag_axis=False,
    )


In [ ]:
if 0:
    import numpy as np
    import torch

    dataset_id = "KDD99"

    feature_dim = int(X.shape[1])
    device = torch.device("cpu")
    num_clusters = 10

    flag_plot_acc = 1

    local_model_params = {
        "feature_dim": feature_dim,
        "num_classes": 1,
        "kappa_n": 0,
        "num_sigma": feature_dim,
        "kappa_join": 1.5,
        "N_r": max(4 * num_clusters, 2 * feature_dim),
        "c_max": 2 * batch_size,
        "kappa_features": 0,
        "device": device,
        "thr_relevance": 1,
    }
    federated_model_params = dict(local_model_params)

    sweep_df, sweep_csv, summary_df, summary_csv, perK_csvs = sweep_clients_all_methods(
        X, y,
        dataset_id=dataset_id,
        exp_id="clientsweep",
        client_grid=[30, 10],
        flag_noniid=True,
        rounds=1,
        num_clusters=num_clusters,
        num_repetitions=20,
        local_model_params=local_model_params,
        federated_model_params=federated_model_params,
        out_root=".benchmarks",
        debug_flag=0,
        flag_save_img=False,
        flag_axis=False,
        save_vec=True,
        overwrite_dataset_csv=True,
        dataset_key_per_K=True,
    )

    print(sweep_csv)
    print(summary_csv)
    display(summary_df)

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

SYNTHETIC_DATASETS = {
    "Aggregation", "Asymmetric", "Compound", "D31",
    "Jain", "Overlap", "Pathbased", "R15",
    "S1", "S2", "S3", "S4",
    "Skewed", "Spiral", "Unbalance", "Unbalance2",
}
METHOD_ORDER = [
    "Our_eFDPC_centers",
    "Our_eFDPC_group",
    "FKDC",           
    "kFED",
    "MUFC",
    "FED_FCM",
    "LLF_FCM",
    "LLF_CM",
]

METHOD_LATEX = {
    "Our_eFDPC_centers": r"eFedDPC${\mathrm{(center)}}$",
    "Our_eFDPC_group":   r"eFedDPC${\mathrm{(group)}}$",
    "FKDC":              r"FKDC",      
    "kFED":              r"kFED",
    "MUFC":              r"MUFC",
    "FED_FCM":           r"FED-FCM",
    "LLF_FCM":           r"LLF-FCM",
    "LLF_CM":            r"LLF-CM",
}

METRICS = ["ACC", "ARI", "NMI", "time_sec"]

METRIC_LATEX = {
    "ACC": "ACC",
    "ARI": "ARI",
    "NMI": "NMI",
    "time_sec": r"Time (s)",
}

METRIC_BEST = {
    "ACC": "max",
    "ARI": "max",
    "NMI": "max",
    "time_sec": "min",
}

METRIC_DECIMALS = {
    "ACC": 3,
    "ARI": 3,
    "NMI": 3,
    "time_sec": 2,
}

REQUIRED_COLS = {"dataset", "split", "method", "ACC", "ARI", "NMI", "time_sec"}

DATASET_INFO_FILENAME = "dataset_info.csv"

def load_all_results(tables_dir: str) -> pd.DataFrame:
    tables_dir = Path(tables_dir)
    files = sorted(tables_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No .csv files found in {tables_dir.resolve()}")

    dfs = []
    for f in files:
        if f.name == DATASET_INFO_FILENAME:
            continue
        d = pd.read_csv(f)
        if REQUIRED_COLS.issubset(set(d.columns)):
            dfs.append(d)

    if not dfs:
        raise FileNotFoundError(
            f"No result CSVs with required cols {sorted(REQUIRED_COLS)} found in {tables_dir.resolve()}"
        )

    out = pd.concat(dfs, ignore_index=True)

    out["time_sec"] = pd.to_numeric(out["time_sec"], errors="coerce")
    return out

def load_dataset_info(tables_dir: str) -> pd.DataFrame:
    p = Path(tables_dir) / DATASET_INFO_FILENAME
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p.resolve()}")

    info = pd.read_csv(p)
    need = {"dataset", "N", "D"}
    if not need.issubset(set(info.columns)):
        raise ValueError(f"{p.name} must contain columns {sorted(need)}")

    info = info.dropna(subset=["dataset"]).copy()
    info["dataset"] = info["dataset"].astype(str)
    info = info.drop_duplicates(subset=["dataset"], keep="last").set_index("dataset")
    info["N"] = info["N"].astype(int)
    info["D"] = info["D"].astype(int)
    return info

def _escape_texttt(s: str) -> str:
    return r"\texttt{" + str(s).replace("_", r"\_") + r"}"

def _dataset_cell_with_info(ds: str, info: pd.DataFrame | None) -> str:
    name_tex = _escape_texttt(ds)
    if info is None or ds not in info.index:
        return name_tex
    N = int(info.loc[ds, "N"])
    D = int(info.loc[ds, "D"])
    return r"\shortstack[l]{" + name_tex + r"\\[-2pt]{\scriptsize (" + f"{N}/{D}" + r")}}"

def _order_methods(g: pd.DataFrame):
    present = list(g["method"].unique())
    ordered = [m for m in METHOD_ORDER if m in present] + [m for m in present if m not in METHOD_ORDER]
    g = g.copy()
    g["method"] = pd.Categorical(g["method"], categories=ordered, ordered=True)
    return g, ordered

def aggregate_mean_std(df_all: pd.DataFrame, split="noniid"):
    df = df_all[df_all["split"] == split].copy()
    if df.empty:
        raise ValueError(f"No rows for split='{split}'")

    df["time_sec"] = pd.to_numeric(df["time_sec"], errors="coerce")

    g = df.groupby(["dataset", "method"], as_index=False)[METRICS].agg(["mean", "std"])
    g.columns = ["dataset", "method"] + [f"{m}_{s}" for m, s in g.columns.tolist()[2:]]
    g, method_order = _order_methods(g)
    ds_order = list(dict.fromkeys(df["dataset"].tolist()))
    return g, ds_order, method_order

def _build_latex_table_for_datasets(
    df_all: pd.DataFrame,
    dataset_info: pd.DataFrame | None,
    ds_list: list[str],
    split="noniid",
    caption=None,
    label="tab:clustering",
    placement="!ht",
    decimals=3,
    std_fill_if_nan=0.0,
    show_std: int = 1,
) -> str:
    if caption is None:
        caption = (
            "Clustering comparison (noniid split): mean $\\pm$ std."
            if show_std else
            "Clustering comparison (noniid split): mean."
        )

    g, ds_order_all, method_order = aggregate_mean_std(df_all, split=split)

    ds_set = set(ds_order_all)
    ds_order = [ds for ds in ds_list if ds in ds_set]

    method_headers = [METHOD_LATEX.get(m, m) for m in method_order]
    col_format = "l|l|" + "c" * len(method_headers)

    piv_mean, piv_std = {}, {}
    for metric in METRICS:
        piv_mean[metric] = (
            g.pivot_table(
                index="dataset", columns="method", values=f"{metric}_mean",
                aggfunc="mean", observed=False
            )
            .reindex(index=ds_order_all)
            .reindex(columns=method_order)
        )
        if show_std:
            piv_std[metric] = (
                g.pivot_table(
                    index="dataset", columns="method", values=f"{metric}_std",
                    aggfunc="mean", observed=False
                )
                .reindex(index=ds_order_all)
                .reindex(columns=method_order)
            )

    formatted = {}
    for metric in METRICS:
        mean_mat = piv_mean[metric]

        if METRIC_BEST.get(metric, "max") == "min":
            best_val = mean_mat.min(axis=1, skipna=True)
        else:
            best_val = mean_mat.max(axis=1, skipna=True)

        if show_std:
            std_mat = piv_std[metric].copy().fillna(std_fill_if_nan)

        fmt = pd.DataFrame(index=mean_mat.index, columns=mean_mat.columns, dtype=object)
        for ds in fmt.index:
            for m in fmt.columns:
                mu = mean_mat.loc[ds, m]
                if pd.isna(mu):
                    cell = "--"
                else:
                    d = METRIC_DECIMALS.get(metric, decimals)
                    if show_std:
                        sd = std_mat.loc[ds, m]
                        cell = f"{float(mu):.{d}f} $\\pm$ {float(sd):.{d}f}"
                    else:
                        cell = f"{float(mu):.{d}f}"

                    if pd.notna(best_val.loc[ds]) and np.isclose(float(mu), float(best_val.loc[ds])):
                        cell = r"\textbf{" + cell + "}"
                fmt.loc[ds, m] = cell

        formatted[metric] = fmt

    lines = []
    lines.append(r"\begin{table*}[" + placement + r"]")
    lines.append(r"  \centering")
    lines.append(r"  \caption{" + caption + r"}\label{" + label + r"}")
    lines.append(r"  \begin{tabular}{" + col_format + r"}")
    lines.append(r"    \toprule")
    lines.append(r"    Dataset & Metric & " + " & ".join(method_headers) + r" \\")
    lines.append(r"    \midrule")

    for i, ds in enumerate(ds_order):
        ds_tex = _dataset_cell_with_info(ds, dataset_info)
        for r_idx, metric in enumerate(METRICS):
            metric_label = METRIC_LATEX.get(metric, metric)
            row_vals = [formatted[metric].loc[ds, m] for m in method_order]
            left = r"\multirow{" + str(len(METRICS)) + r"}{*}{" + ds_tex + r"}" if r_idx == 0 else ""
            lines.append("    " + left + " & " + metric_label + " & " + " & ".join(row_vals) + r" \\")
        if i != len(ds_order) - 1:
            lines.append(r"    \midrule")

    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"\end{table*}")

    return "\n".join(lines)

def latex_two_tables_synth_real(
    df_all: pd.DataFrame,
    dataset_info: pd.DataFrame | None,
    split="noniid",
    decimals=3,
    show_std: int = 1,
    std_fill_if_nan=0.0,
    placement="!ht",
    synth_caption=None,
    real_caption=None,
    synth_label="tab:clustering_synth",
    real_label="tab:clustering_real",
) -> str:
    _, ds_order_all, _ = aggregate_mean_std(df_all, split=split)

    ds_synth = [ds for ds in ds_order_all if ds in SYNTHETIC_DATASETS]
    ds_real  = [ds for ds in ds_order_all if ds not in SYNTHETIC_DATASETS]

    if synth_caption is None:
        synth_caption = (
            "Clustering comparison on synthetic datasets (noniid split): mean $\\pm$ std."
            if show_std else
            "Clustering comparison on synthetic datasets (noniid split): mean."
        )
    if real_caption is None:
        real_caption = (
            "Clustering comparison on real-world datasets (noniid split): mean $\\pm$ std."
            if show_std else
            "Clustering comparison on real-world datasets (noniid split): mean."
        )

    tex_synth = _build_latex_table_for_datasets(
        df_all=df_all,
        dataset_info=dataset_info,
        ds_list=ds_synth,
        split=split,
        caption=synth_caption,
        label=synth_label,
        placement=placement,
        decimals=decimals,
        std_fill_if_nan=std_fill_if_nan,
        show_std=show_std,
    )

    tex_real = _build_latex_table_for_datasets(
        df_all=df_all,
        dataset_info=dataset_info,
        ds_list=ds_real,
        split=split,
        caption=real_caption,
        label=real_label,
        placement=placement,
        decimals=decimals,
        std_fill_if_nan=std_fill_if_nan,
        show_std=show_std,
    )

    return tex_synth + "\n\n" + tex_real

TABLES_DIR = ".benchmarks/tables"
df_all = load_all_results(TABLES_DIR)
info = load_dataset_info(TABLES_DIR)

latex = latex_two_tables_synth_real(
    df_all,
    dataset_info=info,
    split="noniid",
    show_std=0,
    decimals=3,
    synth_label="tab:clustering_synth",
    real_label="tab:clustering_real",
)

print(latex)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SYNTHETIC_DATASETS = {
    "Aggregation", "Asymmetric", "Compound", "D31",
    "Jain", "Overlap", "Pathbased", "R15", "Flame",
    "S1", "S2", "S3", "S4",
    "Skewed", "Spiral", "Unbalance", "Unbalance2",
}

METHOD_ORDER = [
    "Our_eFDPC_centers",
    "Our_eFDPC_group",
    "FKDC",
    "kFED",
    "MUFC",
    "FED_FCM",
    "LLF_FCM",
    "LLF_CM",
]

METHOD_LATEX = {
    "Our_eFDPC_centers": r"eFedDPC-$\mathcal{P}$",
    "Our_eFDPC_group":   r"eFedDPC-$\mathcal{G}$",
    "FKDC":              r"FKDC",
    "kFED":              r"kFED",
    "MUFC":              r"MUFC",
    "FED_FCM":           r"FED-FCM",
    "LLF_FCM":           r"LLF-FCM",
    "LLF_CM":            r"LLF-CM",
}

METRICS = ["ACC", "ARI", "NMI", "time_sec"]

METRIC_LATEX = {
    "ACC": "ACC",
    "ARI": "ARI",
    "NMI": "NMI",
    "time_sec": "Time",
}

METRIC_BEST = {
    "ACC": "max",
    "ARI": "max",
    "NMI": "max",
    "time_sec": "min",
}

METRIC_DECIMALS = {
    "ACC": 3,
    "ARI": 3,
    "NMI": 3,
    "time_sec": 2,
}

REQUIRED_COLS = {"dataset", "split", "method", "ACC", "ARI", "NMI", "time_sec"}
DATASET_INFO_FILENAME = "dataset_info.csv"

def load_all_results(tables_dir: str) -> pd.DataFrame:
    tables_dir = Path(tables_dir)
    files = sorted(tables_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No .csv files found in {tables_dir.resolve()}")
    dfs = []
    for f in files:
        if f.name == DATASET_INFO_FILENAME:
            continue
        d = pd.read_csv(f)
        if REQUIRED_COLS.issubset(set(d.columns)):
            dfs.append(d)
    if not dfs:
        raise FileNotFoundError(
            f"No result CSVs with required cols {sorted(REQUIRED_COLS)} found in {tables_dir.resolve()}"
        )
    out = pd.concat(dfs, ignore_index=True)
    out["time_sec"] = pd.to_numeric(out["time_sec"], errors="coerce")
    return out

def load_dataset_info(tables_dir: str) -> pd.DataFrame:
    p = Path(tables_dir) / DATASET_INFO_FILENAME
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p.resolve()}")
    info = pd.read_csv(p)
    need = {"dataset", "N", "D"}
    if not need.issubset(set(info.columns)):
        raise ValueError(f"{p.name} must contain columns {sorted(need)}")
    info = info.dropna(subset=["dataset"]).copy()
    info["dataset"] = info["dataset"].astype(str)
    info = info.drop_duplicates(subset=["dataset"], keep="last").set_index("dataset")
    info["N"] = info["N"].astype(int)
    info["D"] = info["D"].astype(int)
    return info

def _escape_texttt(s: str) -> str:
    return r"\texttt{" + str(s).replace("_", r"\_") + r"}"

def _dataset_cell_with_info(ds: str, info: pd.DataFrame | None) -> str:
    name_tex = _escape_texttt(ds)
    if info is None or ds not in info.index:
        return name_tex
    N = int(info.loc[ds, "N"])
    D = int(info.loc[ds, "D"])
    return r"\shortstack[l]{" + name_tex + r"\\[-2pt]{\scriptsize (" + f"{N}/{D}" + r")}}"

def _order_methods(g: pd.DataFrame):
    present = list(g["method"].unique())
    ordered = [m for m in METHOD_ORDER if m in present] + [m for m in present if m not in METHOD_ORDER]
    g = g.copy()
    g["method"] = pd.Categorical(g["method"], categories=ordered, ordered=True)
    return g, ordered

def _fmt_sd_no0(x, d):
    s = f"{float(x):.{d}f}"
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s

def aggregate_mean_std(df_all: pd.DataFrame, split="noniid"):
    df = df_all[df_all["split"] == split].copy()
    if df.empty:
        raise ValueError(f"No rows for split='{split}'")
    df["time_sec"] = pd.to_numeric(df["time_sec"], errors="coerce")
    g = df.groupby(["dataset", "method"], as_index=False)[METRICS].agg(["mean", "std"])
    g.columns = ["dataset", "method"] + [f"{m}_{s}" for m, s in g.columns.tolist()[2:]]
    g, method_order = _order_methods(g)
    ds_order = list(dict.fromkeys(df["dataset"].tolist()))
    return g, ds_order, method_order

def _build_latex_table_for_datasets(
    df_all: pd.DataFrame,
    dataset_info: pd.DataFrame | None,
    ds_list: list[str],
    split="noniid",
    caption=None,
    label="tab:clustering",
    placement="!t",
    decimals=3,
    std_fill_if_nan=0.0,
    show_std: int = 1,
    pm_decimals: int = 2,
    tabcolsep_pt: int = 2,
    arraystretch: float | None = None,
):
    if caption is None:
        caption = "Clustering comparison"

    g, ds_order_all, method_order = aggregate_mean_std(df_all, split=split)
    ds_set = set(ds_order_all)
    ds_order = [ds for ds in ds_list if ds in ds_set]

    method_headers = [METHOD_LATEX.get(m, m) for m in method_order]
    col_format = "l|l|" + ("l" * len(method_headers))

    piv_mean, piv_std = {}, {}
    for metric in METRICS:
        piv_mean[metric] = (
            g.pivot_table(
                index="dataset", columns="method", values=f"{metric}_mean",
                aggfunc="mean", observed=False
            )
            .reindex(index=ds_order_all)
            .reindex(columns=method_order)
        )
        if show_std:
            piv_std[metric] = (
                g.pivot_table(
                    index="dataset", columns="method", values=f"{metric}_std",
                    aggfunc="mean", observed=False
                )
                .reindex(index=ds_order_all)
                .reindex(columns=method_order)
            )

    formatted = {}
    for metric in METRICS:
        mean_mat = piv_mean[metric]
        if METRIC_BEST.get(metric, "max") == "min":
            best_val = mean_mat.min(axis=1, skipna=True)
        else:
            best_val = mean_mat.max(axis=1, skipna=True)

        if show_std:
            std_mat = piv_std[metric].copy().fillna(std_fill_if_nan)

        fmt = pd.DataFrame(index=mean_mat.index, columns=mean_mat.columns, dtype=object)
        for ds in fmt.index:
            for m in fmt.columns:
                mu = mean_mat.loc[ds, m]
                if pd.isna(mu):
                    cell = "--"
                else:
                    if show_std:
                        d = int(pm_decimals)
                        sd = std_mat.loc[ds, m]
                        cell = f"{float(mu):.{d}f}" + r"$\pm$" + _fmt_sd_no0(sd, d)
                    else:
                        d = int(METRIC_DECIMALS.get(metric, decimals))
                        cell = f"{float(mu):.{d}f}"
                    if pd.notna(best_val.loc[ds]) and np.isclose(float(mu), float(best_val.loc[ds])):
                        cell = r"\textbf{" + cell + "}"
                fmt.loc[ds, m] = cell
        formatted[metric] = fmt

    lines = []
    lines.append(r"\begin{table}[" + placement + r"]")
    lines.append(r"  \centering")
    lines.append(r"  \footnotesize")
    lines.append(r"  \caption{" + caption + r"}\label{" + label + r"}")
    lines.append(r"  \setlength{\tabcolsep}{" + str(int(tabcolsep_pt)) + r"pt}")
    if arraystretch is not None:
        lines.append(r"  \renewcommand{\arraystretch}{" + str(float(arraystretch)) + r"}")
    lines.append(r"  \begin{tabular}{" + col_format + r"}")
    lines.append(r"    \toprule")
    lines.append(r"    Dataset & Metric & " + " & ".join(method_headers) + r" \\")
    lines.append(r"    \midrule")

    for i, ds in enumerate(ds_order):
        ds_tex = _dataset_cell_with_info(ds, dataset_info)
        for r_idx, metric in enumerate(METRICS):
            metric_label = METRIC_LATEX.get(metric, metric)
            row_vals = [formatted[metric].loc[ds, m] for m in method_order]
            left = r"\multirow{" + str(len(METRICS)) + r"}{*}{" + ds_tex + r"}" if r_idx == 0 else ""
            lines.append("    " + left + " & " + metric_label + " & " + " & ".join(row_vals) + r" \\")
        if i != len(ds_order) - 1:
            lines.append(r"    \midrule")

    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

def latex_two_tables_synth_real(
    df_all: pd.DataFrame,
    dataset_info: pd.DataFrame | None,
    split="noniid",
    decimals=3,
    show_std: int = 1,
    std_fill_if_nan=0.0,
    placement="!t",
    synth_caption=None,
    real_caption=None,
    synth_label="tab:clustering_synth",
    real_label="tab:clustering_real",
    pm_decimals: int = 2,
    tabcolsep_pt: int = 2,
    arraystretch: float | None = None,
):
    _, ds_order_all, _ = aggregate_mean_std(df_all, split=split)
    ds_synth = [ds for ds in ds_order_all if ds in SYNTHETIC_DATASETS]
    ds_real  = [ds for ds in ds_order_all if ds not in SYNTHETIC_DATASETS]

    if synth_caption is None:
        synth_caption = "Clustering comparison on synthetic datasets"
    if real_caption is None:
        real_caption = "Clustering comparison on real-world datasets"

    tex_synth = _build_latex_table_for_datasets(
        df_all=df_all,
        dataset_info=dataset_info,
        ds_list=ds_synth,
        split=split,
        caption=synth_caption,
        label=synth_label,
        placement=placement,
        decimals=decimals,
        std_fill_if_nan=std_fill_if_nan,
        show_std=show_std,
        pm_decimals=pm_decimals,
        tabcolsep_pt=tabcolsep_pt,
        arraystretch=arraystretch,
    )

    tex_real = _build_latex_table_for_datasets(
        df_all=df_all,
        dataset_info=dataset_info,
        ds_list=ds_real,
        split=split,
        caption=real_caption,
        label=real_label,
        placement=placement,
        decimals=decimals,
        std_fill_if_nan=std_fill_if_nan,
        show_std=show_std,
        pm_decimals=pm_decimals,
        tabcolsep_pt=tabcolsep_pt,
        arraystretch=arraystretch,
    )

    return tex_synth + "\n\n" + tex_real

TABLES_DIR = ".benchmarks/tables"
df_all = load_all_results(TABLES_DIR)
info = load_dataset_info(TABLES_DIR)

latex = latex_two_tables_synth_real(
    df_all,
    dataset_info=info,
    split="noniid",
    show_std=1,
    pm_decimals=2,
    decimals=3,
    tabcolsep_pt=2,
    arraystretch=None,
    synth_label="tab:clustering_synth",
    real_label="tab:clustering_real",
)

print(latex)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_path = r".benchmarks\sweeps\MNIST_C3_R10_eFDPC.csv"
csv_path = r".benchmarks\sweeps\MNIST_C10_R10_eFDPC.csv"
csv_path = r".benchmarks\sweeps\MNIST_C30_R10_eFDPC.csv"
csv_path = r".benchmarks\sweeps\MNIST_C3_R10_eFDPC.csv"
df = pd.read_csv(csv_path)

for c in ["ACC", "ARI", "NMI"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

keep_cols = [c for c in ["ACC", "ARI", "NMI"] if c in df.columns]
df = df.dropna(subset=keep_cols, how="all").reset_index(drop=True)

if "round_idx" in df.columns:
    df["round_idx"] = pd.to_numeric(df["round_idx"], errors="coerce")
    sort_cols = ["round_idx"] + (["method"] if "method" in df.columns else [])
    df = df.sort_values(sort_cols).reset_index(drop=True)

def _get_xy(sub: pd.DataFrame, metric: str):
    if "round_idx" in sub.columns and sub["round_idx"].notna().any():
        x = sub["round_idx"].to_numpy(dtype=float)
    else:
        x = np.arange(1, len(sub) + 1, dtype=float)
    y = sub[metric].to_numpy(dtype=float)
    return x, y

def plot_metric(metric: str, ylabel: str):
    if metric not in df.columns:
        return

    plt.figure()

    if "method" in df.columns:
        for method in sorted(df["method"].dropna().unique()):
            sub = df[df["method"] == method].copy()
            if "round_idx" in sub.columns:
                sub = sub.sort_values("round_idx")
            x, y = _get_xy(sub, metric)
            plt.plot(x, y, marker="o", label=str(method))
        plt.legend()
    else:
        sub = df.copy()
        if "round_idx" in sub.columns:
            sub = sub.sort_values("round_idx")
        x, y = _get_xy(sub, metric)
        plt.plot(x, y, marker="o")

    plt.xlabel("Round")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

plot_metric("ACC", "ACC")

if ("ARI" in df.columns) or ("NMI" in df.columns):
    plt.figure()

    if "method" in df.columns:
        for method in sorted(df["method"].dropna().unique()):
            sub = df[df["method"] == method].copy()
            if "round_idx" in sub.columns:
                sub = sub.sort_values("round_idx")

            if "ARI" in sub.columns:
                x, y = _get_xy(sub, "ARI")
                plt.plot(x, y, marker="o", label=f"{method} (ARI)")
            if "NMI" in sub.columns:
                x, y = _get_xy(sub, "NMI")
                plt.plot(x, y, marker="o", label=f"{method} (NMI)")

        plt.legend()
    else:
        sub = df.copy()
        if "round_idx" in sub.columns:
            sub = sub.sort_values("round_idx")

        if "ARI" in sub.columns:
            x, y = _get_xy(sub, "ARI")
            plt.plot(x, y, marker="o", label="ARI")
        if "NMI" in sub.columns:
            x, y = _get_xy(sub, "NMI")
            plt.plot(x, y, marker="o", label="NMI")

        plt.legend()

    plt.xlabel("Round")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()